This notebook is the **execution and evaluation bench** for the paper **Generating and Assessing Natural Language Descriptions of Multi-Perspective Declarative Process Models** (the *Nisaba* framework). It contains the dependencies, utilities, prompts, and schema for the generation pipeline (MP-Declare → intermediary description → agentic narrative → reconstruction) together with the objective evaluation of the generated descriptions (correctness, completeness, understandability). All LLM calls are routed through **OpenRouter**, a single gateway to multiple LLM providers (OpenAI, Anthropic, DeepSeek, Moonshot, and others).

# 1. Requirements

## Dependencies

In [ ]:
# === Dependencies ===
# Pinned for reproducibility. The generation layer (CrewAI + LiteLLM) runs on Colab;
# torch is pre-installed there.
!pip install -q declare4py==2.2.0 openai==2.45.0 pydantic==2.11.7 scipy==1.18.0 numpy==2.5.1 pandas==3.0.3 networkx==3.6.1 tenacity==9.1.4 sentence-transformers==5.6.0 instructor==1.15.4 textstat==0.7.13 transformers==5.13.0 gdown==6.1.0 "litellm>=1.55" crewai crewai-tools

## Imports

In [ ]:
# === Imports ===
import os, re, json, math, time, glob, zipfile, logging, contextlib
from collections import Counter, defaultdict
from types import SimpleNamespace
from typing import List, Optional, Union, Literal

import numpy as np
import pandas as pd
import networkx as nx
from pydantic import BaseModel, Field, conlist, field_validator, model_validator
from tenacity import retry, wait_random_exponential, stop_after_attempt

from openai import OpenAI                       # OpenAI SDK, pointed at OpenRouter
import litellm, instructor                      # provider-agnostic routing + structured output
from litellm.integrations.custom_logger import CustomLogger
from sentence_transformers import SentenceTransformer   # gte-large embedder (semantic distance)
import textstat                                 # Flesch Reading Ease

from Declare4Py.ProcessModels.DeclareModel import DeclareModel
from crewai import Agent, Task, Process, Crew, LLM       # agentic narrative synthesis
from google.colab import userdata, files

# 2. Setup & Configuration

This section sets up the utilities to reach the LLM providers through **OpenRouter** and configures the environment variables, the default model, and the determinism controls. You need an **OpenRouter API key** with credit; in Colab, store it as a secret named `openrouter_key` (Tools › Secrets). Get a key at [openrouter.ai/keys](https://openrouter.ai/keys).

In [ ]:
# === Environment configuration: OpenRouter unified gateway ===
# One API key / one bill for OpenAI, Anthropic, DeepSeek, Moonshot, and other providers.
# In Colab, add a secret named 'openrouter_key' (Tools > Secrets) with your OpenRouter API key.
seed = 42
os.environ["OPENROUTER_API_KEY"] = userdata.get('openrouter_key')
os.environ["OPENAI_API_KEY"] = os.environ["OPENROUTER_API_KEY"]   # some libraries read OPENAI_API_KEY

# Default model used only when a call does not pass model= (before set_active_model). Routed via OpenRouter.
DEFAULT_MODEL = "openrouter/deepseek/deepseek-v4-flash"   # default generator for the single-model demo

# OpenAI-SDK client pointed at OpenRouter.
client = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")

# Embedder for Semantic Distance: gte-large (paper's embedder, tau=0.2) with a position_ids buffer repair
# (some transformers versions leave it corrupted after the RoPE meta-load) and a MiniLM fallback (tau=0.5).
def load_embedder():
    def _fix_position_ids(model):
        try:
            import torch
            for mod in model.modules():
                buf = getattr(mod, "position_ids", None)
                if buf is not None and hasattr(buf, "shape"):
                    fixed = torch.arange(buf.shape[-1]).reshape(buf.shape)
                    mod.register_buffer("position_ids", fixed, persistent=False)
        except Exception as e:
            print("    [embedder] position_ids fix skipped:", e)
    try:
        m = SentenceTransformer("Alibaba-NLP/gte-large-en-v1.5", trust_remote_code=True)
        try:
            m.encode(["warmup sentence"])
        except Exception:
            _fix_position_ids(m)
            m.encode(["warmup sentence"])
        return m, 0.2   # tau calibrated for gte-large
    except Exception as e:
        print("    [embedder] gte-large failed, falling back to MiniLM:", str(e)[:200])
        return SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2"), 0.5

embedding_model, EMB_TAU = load_embedder()

### Multi-provider model configuration (PM-LLM-Benchmark v2.2)

To keep the evaluation vendor-neutral, descriptions are generated across four providers chosen from the PM-LLM-Benchmark v2.2 leaderboard: two proprietary (OpenAI, Anthropic) and two open-weight and self-hostable (DeepSeek, Moonshot). Model ids below are LiteLLM routes over OpenRouter; set the matching API keys, or point the open-weight ones at a local endpoint (vLLM/Ollama) for on-prem runs.

In [ ]:
# === Model configuration (PM-LLM-Benchmark v2.2) ===
# Four generators (2 proprietary + 2 open-weight) produce the descriptions WITHOUT reasoning; two readers
# (1 proprietary + 1 open-weight) answer the comprehension QA. Every generator scores BELOW the weakest
# reader on the benchmark, so each description is assessed by models strictly more capable than the one
# that produced it, and the reader providers are disjoint from the generator providers (no same-provider
# self-preference). Routed via OpenRouter -> litellm "openrouter/<slug>".
#
#   Role       Provider            Model               PM-LLM v2.2 score
#   generator  Anthropic (closed)  claude-sonnet-5     32.6
#   generator  xAI       (closed)  grok-4.20           32.3
#   generator  DeepSeek  (open)    deepseek-v4-flash   33.6
#   generator  Z-AI      (open)    glm-5.2             33.8
#   reader     OpenAI    (closed)  gpt-5.6-terra       34.9
#   reader     Moonshot  (open)    kimi-k2.5           34.0
#   => max(generator) = 33.8 < min(reader) = 34.0 : readers are strictly stronger than every generator.
GENERATORS = {
    "anthropic": "openrouter/anthropic/claude-sonnet-5",
    "xai":       "openrouter/x-ai/grok-4.20",
    "deepseek": "openrouter/deepseek/deepseek-v4-flash",
    "zai":      "openrouter/z-ai/glm-5.2",
}
READERS = {
    "openai":   "openrouter/openai/gpt-5.6-terra",
    "moonshot": "openrouter/moonshotai/kimi-k2.5",
}
PROVIDER_OF = {m: p for p, m in {**GENERATORS, **READERS}.items()}
# Per-model reasoning control: every generator AND reader runs with reasoning disabled (so the readers'
# benchmark scores -- and therefore the generator < reader ordering -- are the no-reasoning ones).
MODEL_PARAMS = {m: {"reasoning_effort": "none"} for m in {**GENERATORS, **READERS}.values()}
# Verify exact OpenRouter slugs and per-model support with capability_report() before a full run.

### Declare4Py model helpers

In [ ]:
# Extend DeclareModel (declare4py) with helpers to list attributes and binds, and configure a logger.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    force=True,
    handlers=[logging.StreamHandler()],
)
logger = logging.getLogger("Nisaba")

class ExtendedDeclareModel(DeclareModel):
    """DeclareModel (declare4py) extended with helpers to return the model's attributes and binds."""

    def get_decl_attributes(self):
        attributes = []
        for attr_name, attr_obj in self.parsed_model.attributes_list.items():
            attr_value = ""
            if attr_obj.attr_value:
                attr_value = ": " + attr_obj.attr_value.value_original
            attributes.append(f"{attr_name}{attr_value}")
        return attributes

    def get_decl_binds(self):
        binds = []
        for event_type, events in self.parsed_model.events.items():
            for event_name, event_obj in events.items():
                if event_obj.attributes:
                    bound_attrs = ", ".join(a.get_name() for a in event_obj.attributes.values())
                    binds.append(f"bind {event_name}: {bound_attrs}")
        return binds

### Provider-agnostic generation layer (LiteLLM + instructor)

A thin, provider-agnostic layer over **LiteLLM** (a unified OpenAI-format gateway to many providers) and **instructor** (portable Pydantic `response_model` with validation and repair) exposes two primitives: `chat_completion_request` (plain text and **function/tool calling**, used for the intermediary step) and `structured_output` (**structured Pydantic output**, used for reconstruction). The active model is selected with `set_active_model`; tool calls and structured outputs are expressed once and run on any provider (native where supported, JSON-mode fallback otherwise).

In [ ]:
# === Provider-agnostic, capability-aware LLM primitives (LiteLLM + instructor) + call logging ===

litellm.drop_params = True                 # backstop: a provider that lacks a param drops it instead of erroring
litellm.enable_json_schema_validation = True

# --- Determinism configuration (single source of truth, shared by every provider) ---
GEN_SEED = seed            # 42 (from the config cell)
GEN_TEMPERATURE = 0.0      # greedy decoding where supported; the pipeline fixes temperature at 0
GEN_TOP_P = 1.0            # set explicitly so sampling is not left to provider defaults
GEN_TIMEOUT = 240          # per-call hard timeout (seconds)
GEN_MAX_TOKENS = 8000      # cap on completion length

PROVIDER_CONFIG = {
    "openai":   {},                                             # uses OPENAI_API_KEY
    "google":   {},                                             # uses GEMINI_API_KEY (litellm route gemini/...)
    "deepseek": {},                                             # uses DEEPSEEK_API_KEY (litellm route deepseek/...)
    "qwen":     {"api_base": os.environ.get("QWEN_BASE_URL", "")},  # DashScope OpenAI-compat or local vLLM
}

# ------------------------------------------------------------------------------------------------
# COMPLETE PROVENANCE: one JSONL record per LLM call, capturing EVERY request+response in the run.
# Because CrewAI also calls LiteLLM under the hood, a single global LiteLLM callback captures the
# generation, reconstruction AND CrewAI narrative steps -- not only calls that go through our own primitives.
# Each record stores: step context, model, the params actually sent, which params were dropped, the
# FULL prompt messages, and the FULL response (content + any tool calls): an auditable,
# replayable trace of the full run.
# ------------------------------------------------------------------------------------------------
LLM_CALL_LOG = "llm_calls.jsonl"

# Ambient step context, merged into every logged call. Set it around a step with `with log_context(...)`.
LOG_CONTEXT = {}
@contextlib.contextmanager
def log_context(**kw):
    global LOG_CONTEXT
    prev = LOG_CONTEXT
    LOG_CONTEXT = {**prev, **kw}
    try:
        yield
    finally:
        LOG_CONTEXT = prev

def set_step(**kw):
    """Set the ambient provenance step for the following logged calls (linear-notebook convenience)."""
    global LOG_CONTEXT
    LOG_CONTEXT = dict(kw)

def _extract_response(response_obj):
    try:
        msg = response_obj.choices[0].message
        out = {"content": getattr(msg, "content", None)}
        tcs = getattr(msg, "tool_calls", None)
        if tcs:
            out["tool_calls"] = [{"name": t.function.name, "arguments": t.function.arguments} for t in tcs]
        return out
    except Exception:
        try: return {"raw": response_obj.model_dump()}
        except Exception: return {"raw": str(response_obj)[:4000]}

def _extract_usage(response_obj):
    """Token usage for the cost report; litellm exposes response.usage on a successful call."""
    try:
        u = getattr(response_obj, "usage", None)
        if u is None:
            return None
        return {"prompt_tokens": getattr(u, "prompt_tokens", None),
                "completion_tokens": getattr(u, "completion_tokens", None),
                "total_tokens": getattr(u, "total_tokens", None)}
    except Exception:
        return None

class _NisabaCallLogger(CustomLogger):
    """Global LiteLLM logger: writes one full record per call to LLM_CALL_LOG."""
    def _write(self, kwargs, response_obj, status, start_time, end_time):
        try:
            lp = kwargs.get('litellm_params') or {}
            rec = {
                "t": time.time(), "status": status, **LOG_CONTEXT,
                "model": kwargs.get("model"),
                "params_sent": kwargs.get("optional_params", {}),
                "call_metadata": lp.get("metadata") or {},
                "messages": kwargs.get("messages"),
                "response": _extract_response(response_obj) if response_obj is not None else None,
                "usage": _extract_usage(response_obj) if response_obj is not None else None,
            }
            try: rec['duration_s'] = (end_time - start_time).total_seconds()
            except Exception: rec['duration_s'] = None
            with open(LLM_CALL_LOG, "a", encoding="utf-8") as f:
                f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")
        except Exception as e:
            print(f"[call-logger] {e}")
    def log_success_event(self, kwargs, response_obj, start_time, end_time):
        self._write(kwargs, response_obj, "success", start_time, end_time)
    def log_failure_event(self, kwargs, response_obj, start_time, end_time):
        self._write(kwargs, response_obj, "failure", start_time, end_time)
    async def async_log_success_event(self, kwargs, response_obj, start_time, end_time):
        self._write(kwargs, response_obj, "success", start_time, end_time)
    async def async_log_failure_event(self, kwargs, response_obj, start_time, end_time):
        self._write(kwargs, response_obj, "failure", start_time, end_time)

litellm.callbacks = [_NisabaCallLogger()]   # captures ALL litellm calls, including CrewAI's narrative agent

# --- Capability probing: send only sampling params a model supports (reasoning models reject them) ---
def _supported_openai_params(model):
    try:
        return set(litellm.get_supported_openai_params(model=model) or [])
    except Exception:
        return None

def _sampling_params(model, temperature, top_p, seed):
    candidate = {"temperature": temperature, "top_p": top_p, "seed": seed}
    sup = _supported_openai_params(model)
    if sup is None:
        return candidate, []
    return {k: v for k, v in candidate.items() if k in sup}, [k for k in candidate if k not in sup]

def supports_tools(model):
    try:
        return bool(litellm.supports_function_calling(model=model))
    except Exception:
        return False

ACTIVE_MODEL = None
def set_active_model(model_id):
    global ACTIVE_MODEL
    ACTIVE_MODEL = model_id

def _resolve_model(model):
    return model or ACTIVE_MODEL or DEFAULT_MODEL

def _provider_kwargs(model):
    cfg = PROVIDER_CONFIG.get(PROVIDER_OF.get(model, ''), {})
    return {k: v for k, v in cfg.items() if v}

def _reasoning_body(model):
    """Map MODEL_PARAMS reasoning_effort onto OpenRouter's reasoning control (sent via extra_body),
    so reasoning is disabled explicitly rather than relying on how LiteLLM reads a raw value."""
    eff = MODEL_PARAMS.get(model, {}).get("reasoning_effort")
    if not eff:        return {}
    if eff == "none":  return {"extra_body": {"reasoning": {"enabled": False}}}
    return {"extra_body": {"reasoning": {"effort": eff}}}

_structured_client = instructor.from_litellm(litellm.completion)

@retry(wait=wait_random_exponential(multiplier=1, max=40), stop=stop_after_attempt(3))
def chat_completion_request(messages, temperature=GEN_TEMPERATURE, seed=None, function_call=None, tools=None, tool_choice=None, model=None, top_p=None):
    model = _resolve_model(model)
    seed = GEN_SEED if seed is None else seed
    top_p = GEN_TOP_P if top_p is None else top_p
    kept, dropped = _sampling_params(model, temperature, top_p, seed)
    response = litellm.completion(
        model=model, messages=messages, tools=tools, tool_choice=tool_choice,
        timeout=GEN_TIMEOUT, max_tokens=GEN_MAX_TOKENS,
        metadata={"dropped_params": dropped}, **kept, **_reasoning_body(model), **_provider_kwargs(model),
    )
    # when plain text is expected, empty content signals a failed call (e.g. a reasoning model spent its
    # token budget on hidden reasoning); raise so @retry re-issues it.
    message = response.choices[0].message
    if tools is None and not getattr(message, "content", None):
        raise RuntimeError(f"empty content from {model}")
    return response

@retry(wait=wait_random_exponential(multiplier=1, max=40), stop=stop_after_attempt(3))
def structured_output(messages, temperature=GEN_TEMPERATURE, seed=None, response_format=None, model=None, top_p=None):
    model = _resolve_model(model)
    seed = GEN_SEED if seed is None else seed
    top_p = GEN_TOP_P if top_p is None else top_p
    kept, dropped = _sampling_params(model, temperature, top_p, seed)
    parsed = _structured_client.chat.completions.create(
        model=model, messages=messages, response_model=response_format, max_retries=3,
        timeout=GEN_TIMEOUT, max_tokens=GEN_MAX_TOKENS,
        metadata={"dropped_params": dropped}, **kept, **_reasoning_body(model), **_provider_kwargs(model),
    )
    message = SimpleNamespace(parsed=parsed, content=parsed.model_dump_json())
    return SimpleNamespace(choices=[SimpleNamespace(message=message)])

def capability_report():
    """Print per-model support for sampling params, tools and response schema. Run once (after setting
    API keys) to VERIFY the layer is valid for all four providers."""
    print(f"{'role':6} {'provider':9} {'model':32} {'sampling':40} tools schema")
    for role, d in [("GEN", GENERATORS), ("READER", READERS)]:
        for p, m in d.items():
            sup = _supported_openai_params(m)
            sp = ",".join(sorted((sup or set()) & {"temperature", "top_p", "seed", "response_format"})) or "(unknown)"
            try: schema = litellm.supports_response_schema(model=m)
            except Exception: schema = None
            print(f"{role:6} {p:9} {m:32} {sp:40} {str(supports_tools(m)):5} {schema}")


### MP-Declare data-model schema

The following data model were used to validate if MP-Declare models are compliant with Declare4Py specifications.

In [ ]:
# Structured Output Data Schema
class Activity(BaseModel):
    """
    Represents an activity in a process model.

    Attributes:
        name (str): The name of the activity.
        description (str): A detailed description of what the activity entails.
    """
    name: str = Field(
        ...,
        description="The name of the activity. It's represented by an action in the process.",
    )
    description: str = Field(
        ...,
        description="A detailed description of what the activity entails.",
    )

class Attribute(BaseModel):
    """
    Represents an attribute associated with an activity.

    Attributes:
        type (str): The type of attribute. It can be integer, float or enumeration.
        name (str): The name of the attribute.
        description (str): A detailed description of what the attribute represents.
        min_value (Optional[Union[float, int]]): Minimum value for integer or float attributes.
        max_value (Optional[Union[float, int]]): Maximum value for integer or float attributes.
        enumeration_values (Optional[List[str]]): Possible values for enumeration attributes.
    """
    type: Literal["integer", "float", "enumeration"] = Field(
        ...,
        description="The type of attribute. It can be integer, float or enumeration.",
    )
    name: str = Field(
        ...,
        description="The name of the attribute.",
    )
    description: str = Field(
        ...,
        description="A detailed description of what the attribute represents.",
    )
    min_value: Optional[Union[float, int]] = Field(
        None,
        description="Minimum value for integer or float attributes. Should be <= than max_value."
    )
    max_value: Optional[Union[float, int]] = Field(
        None,
        description="Maximum value for integer or float attributes. Should be >= than min_value."
    )
    enumeration_values: Optional[List[str]] = Field(
        None,
        description="Possible values for enumeration attributes."
    )

    @field_validator('description', mode='before')
    def validate_description(cls, value):
        """
        Ensure description is not empty.

        Raises:
            ValueError: If description is empty.

        Returns:
            value: The validated value.
        """
        logger.debug(f"Validating description for attribute: {value}")
        if not value:
            raise ValueError("Description must not be empty.")
        return value

    @field_validator('min_value', 'max_value', mode='before')
    def check_min_max_values(cls, value, info):
        """
        Validate min_value and max_value based on the type of attribute.

        Raises:
            ValueError: If min_value or max_value are required but not provided.

        Returns:
            value: The validated value.
        """
        logger.debug(f"Validating {info.field_name} for attribute type {info.data['type']}.")
        if info.data['type'] == 'integer':
            if value is not None:
                value = int(value)
            else:
                raise ValueError(f"{info.field_name} is required for {info.data['type']} type.")
        elif info.data['type'] == 'float':
            if value is None:
                raise ValueError(f"{info.field_name} is required for {info.data['type']} type.")
        return value

    @field_validator('min_value', 'max_value', mode='after')
    def ensure_min_less_than_max(cls, value, info):
        """
        Ensure that min_value is less than or equal to max_value.

        Raises:
            ValueError: If min_value is greater than max_value.

        Returns:
            value: The validated value.
        """
        logger.debug(f"Ensuring {info.field_name} for attribute type {info.data['type']} is within valid range.")
        if info.data.get('min_value') is not None and info.data.get('max_value') is not None:
            if info.data['min_value'] > info.data['max_value']:
                raise ValueError("min_value must be less than or equal to max_value.")
        return value

    @field_validator('enumeration_values', mode='before')
    def check_enumeration_values(cls, value, info):
        """
        Validate enumeration_values for enumeration type attributes.

        Raises:
            ValueError: If enumeration_values has fewer than 1 unique items.

        Returns:
            value: The validated value.
        """
        logger.debug(f"Validating enumeration values for attribute type {info.data['type']}.")
        if info.data['type'] == 'enumeration':
            if not value or len(value) < 1:
                raise ValueError("enumeration_values must have at least 1 unique item for enumeration type.")
        return value

class Bind(BaseModel):
    """
    Represents the binding of attributes to an activity.

    Attributes:
        activity (Activity): The activity involved in the binding.
        attributes (List[Attribute]): List of attribute objects bound to the activity.
        description (str): Description of the purpose or role of the bind.
    """
    activity: Activity = Field(
        ...,
        description="The activity involved in the binding."
    )
    attributes: conlist(Attribute) = Field(
        ...,
        description="List of attribute objects bound to the activity. The attribute must have the min_value and max_value if it is integer or float and if it is enumeration it must have at least one option."
    )
    description: str = Field(
        ...,
        description="Description of the purpose or role of the bind."
    )

    @model_validator(mode='before')
    def validate_attributes(cls, values):
        """
        Ensure attributes are valid instances of Attribute.

        Raises:
            ValueError: If attributes are not valid instances of Attribute.

        Returns:
            values: The validated values.
        """
        logger.debug(f"Validating attributes for bind: {values}")
        attributes = values.get('attributes', [])
        for attribute in attributes:
            if not isinstance(attribute, Attribute):
                Attribute(**attribute)
        return values

    @field_validator('description', mode='before')
    def validate_description(cls, value):
        """
        Ensure description is not empty.

        Raises:
            ValueError: If description is empty.

        Returns:
            value: The validated value.
        """
        logger.debug(f"Validating description for bind: {value}")
        if not value:
            raise ValueError("Description must not be empty.")
        return value

class Constraint(BaseModel):
    """
    Represents a constraint in the process model.

    Attributes:
        type (str): The type of constraint. It can be unary or binary.
        description (str): A detailed description of what the constraint entails.
        template (str): The template of the constraint.
        activation (Optional[Activity]): The activation activity involved in the constraint.
        target (Optional[Activity]): The target activity involved in the constraint.
        activation_condition (Optional[str]): Condition to activate the constraint.
        correlation_condition (Optional[str]): Condition to correlate the constraint.
        time_condition (Optional[str]): Condition to specify time constraints.
        cardinality (Optional[int]): The cardinality for the Existence, Absence, and Exactly templates.
        all_binds (Dict[str, Bind]): Dictionary of binds associated with the constraint.
    """
    type: Literal["unary", "binary"] = Field(
        ...,
        description="The type of constraint. It can be unary or binary. Unary templates have only activation and binary templates must have activation and target.",
    )
    description: str = Field(
        ...,
        description="A detailed description of what the constraint entails.The time conditions should always be described in the original measure."
    )
    template: Literal["Init", "End", "Existence", "Absence", "Exactly", "Alternate Precedence", "Alternate Response", "Alternate Succession", "Chain Precedence", "Chain Response", "Chain Succession", "Co-Existence", "Precedence", "Responded Existence", "Response", "Succession", "Not Chain Succession", "Not Co-Existence", "Not Succession", "Not Chain Precedence", "Not Chain Response", "Not Precedence", "Not Responded Existence", "Not Response", "Choice", "Exclusive Choice"] = Field(
        ...,
        description="The template of the constraint. If the type of the constraint is unary, the following templates are applicable: Init, End, Existence, Absence, Exactly. If it is binary template, the following are applicable: Alternate Precedence, Alternate Response, Alternate Succession, Chain Precedence, Chain Response, Chain Succession, Co-Existence, Precedence, Responded Existence, Response, Succession, Not Chain Succession, Not Co-Existence, Not Succession, Not Chain Precedence, Not Chain Response, Not Precedence, Not Responded Existence, Not Response, Choice, Exclusive Choice ",
    )
    activation: Activity = Field(
        ...,
        description="The activation activity involved in the constraint. An activation activity is the event or activity that triggers a specific constraint or rule in the process model. It is the starting point or the condition that needs to be met for a constraint to be evaluated."
    )
    target: Optional[Activity] = Field(
        None,
        description="The target activity involved in the constraint. A target activity is the event or activity that is affected or constrained by the activation activity. It is the activity whose occurrence, timing, or order is governed by the constraint that has been activated. Its mandatory that binary templates have target activity."
    )
    activation_condition: Optional[str] = Field(
        None,
        description="Condition to activate the constraint. It must reference attributes of the activation activity and may use operations in, not in, is, is not for enumeration attributes. For integer and float it can use the following operators: ==, !=, >, >=, <, <= . And conditions can be joined using AND and OR.",
    )
    correlation_condition: Optional[str] = Field(
        None,
        description="Condition to correlate the activation and target of the constraint. It must reference attributes of both activation and target activities and may use operations in, not in, is, is not for enumeration attributes. For integer and float it can use the following operators: ==, !=, >, >=, <, <= . And conditions can be joined using AND and OR.",
    )
    time_condition: Optional[str] = Field(
        None,
        description="Condition to specify time of the activation of the constraint. It can be in seconds (s), minutes (m), hours (h) or days (d).",
    )
    cardinality: Optional[int] = Field(
        None,
        description="The cardinality that is mandatory for some unary templates: Existence, Absence, and Exactly."
    )

    @model_validator(mode='before')
    def validate_template_type_match(cls, values):
        """
        Ensure the template matches the type of the constraint.

        Args:
            values (dict): The values to validate.

        Raises:
            ValueError: If the template is not valid for the given type.

        Returns:
            dict: The validated values.
        """
        constraint_type = values.get('type')
        template = values.get('template')

        unary_templates = ["Init", "End", "Existence", "Absence", "Exactly"]
        binary_templates = ["Alternate Precedence", "Alternate Response", "Alternate Succession", "Chain Precedence", "Chain Response", "Chain Succession", "Co-Existence", "Precedence", "Responded Existence", "Response", "Succession", "Not Chain Succession", "Not Co-Existence", "Not Succession", "Not Chain Precedence", "Not Chain Response", "Not Precedence", "Not Responded Existence", "Not Response", "Choice", "Exclusive Choice"]

        logger.debug(f"Validating constraint: type={constraint_type}, template={template}")

        if constraint_type == "unary" and template not in unary_templates:
            raise ValueError(f"Invalid template '{template}' for unary constraint")
        if constraint_type == "binary" and template not in binary_templates:
            raise ValueError(f"Invalid template '{template}' for binary constraint")
        return values

    @field_validator('activation', 'target', mode='before')
    def check_activation_and_target(cls, value, info):
        """
        Validate activation and target based on the type of constraint.

        Raises:
            ValueError: If activation or target is required but not provided.

        Returns:
            value: The validated value.
        """
        logger.debug(f"Validating {info.field_name} for constraint type {info.data['type']}.")
        if info.data['type'] == 'binary':
            if info.field_name == 'activation' and value is None:
                raise ValueError("activation is required for binary type constraints.")
            if info.field_name == 'target' and value is None:
                raise ValueError("target is required for binary type constraints.")
        if info.data['type'] == 'unary' and info.field_name == 'activation' and value is None:
            raise ValueError("activation is required for unary type constraints.")
        return value



    @field_validator('cardinality', mode='before')
    def check_cardinality(cls, value, info):
        """
        Validate cardinality for Existence, Absence and Exactly templates.

        Raises:
            ValueError: If cardinality is required but not provided.

        Returns:
            value: The validated value.
        """
        logger.debug(f"Validating cardinality for template {info.data['template']}.")
        if info.data['template'] in ['Existence', 'Absence', 'Exactly'] and value is None:
            raise ValueError("cardinality is required for Existence, Absence, and Exactly templates.")
        return value


class MPDeclareModel(BaseModel):
    """
    Represents a process model with activities, attributes, binds, and constraints.

    Attributes:
        activities (List[Activity]): List of activities in the process model.
        attributes (List[Attribute]): List of attributes used in the process model.
        binds (List[Bind]): List of binds that links activities and attributes used in the process model.
        constraints (List[Constraint]): List of constraints that rule the process execution.
    """
    activities: List[Activity] = Field(
        ...,
        description="List of activities used in the process model.",
    )
    attributes: List[Attribute] = Field(
        ...,
        description="List of attributes used in the process model.",
    )
    binds: List[Bind] = Field(
        ...,
        description="List of binds that links activities and attributes used in the process model.",
    )
    constraints: List[Constraint] = Field(
        ...,
        description="List of constraints that rule the process execution.",
    )

    def convert_to_string(self) -> str:
        """
        Convert the process model to a string representation.

        Returns:
            str: The string representation of the process model.
        """
        def _nm(x):
            return getattr(x, "name", None) if x is not None else None
        activities_str = "\n".join(
            f"activity {_nm(a)}" for a in (self.activities or []) if _nm(a)
        )
        binds_str = "\n".join(
            f"bind {_nm(b.activity)}: " + ", ".join(_nm(at) for at in (b.attributes or []) if _nm(at))
            for b in (self.binds or []) if b is not None and _nm(getattr(b, "activity", None))
        )
        attributes_str = ""
        for attribute in (self.attributes or []):
            if attribute is None or not _nm(attribute):
                continue
            if attribute.type in ['integer', 'float']:
                attributes_str += f"{attribute.name}: {attribute.type} between {attribute.min_value} and {attribute.max_value}\n"
            elif attribute.type == 'enumeration':
                enumeration_values = ", ".join(attribute.enumeration_values or [])
                attributes_str += f"{attribute.name}: {enumeration_values}\n"
        constraints_str = ""
        for constraint in (self.constraints or []):
            if constraint is None:
                continue
            constraint_str = ""
            an = _nm(getattr(constraint, "activation", None))
            if constraint.type == 'unary':
                if not an:
                    continue
                if constraint.cardinality and constraint.cardinality > 1:
                    constraint_str = f"{constraint.template}{constraint.cardinality}[{an}]"
                else:
                    constraint_str = f"{constraint.template}[{an}]"
                constraint_str += f" |{constraint.activation_condition or ''} |{constraint.time_condition or ''}"
            elif constraint.type == 'binary':
                tn = _nm(getattr(constraint, "target", None))
                if not an or not tn:
                    continue
                constraint_str = f"{constraint.template}[{an}, {tn}]"
                constraint_str += f" |{constraint.activation_condition or ''} |{constraint.correlation_condition or ''} |{constraint.time_condition or ''}"
            if constraint_str:
                constraints_str += f"{constraint_str}\n"
        final_str = "\n".join([activities_str, binds_str, attributes_str.strip(), constraints_str.strip()])
        return final_str


# 3. Nisaba Pipeline

## Step 1 — Model pre-processing & prompt generation

In [ ]:
model ="""
activity Identify Threat
activity Analyze Threat
activity Prioritize Threat
activity Mitigate Threat
activity Report Threat
activity Review Threat
bind Identify Threat: Threat Status
bind Analyze Threat: Threat Level, Impact Score
bind Prioritize Threat: Threat Level
bind Mitigate Threat: Mitigation Strategy
bind Report Threat: Threat Status
bind Review Threat: Response Time
Threat Level: integer between 1 and 10
Impact Score: float between 0.0 and 100.0
Threat Status: Identified, Analyzed, Mitigated, Reported
Mitigation Strategy: Isolation, Patch, Monitor
Response Time: integer between 1 and 72
Init[Identify Threat] | |
End[Review Threat] | |
Chain Response[Identify Threat, Analyze Threat] | | |0,2,h
Response[Analyze Threat, Prioritize Threat] | | |
Precedence[Mitigate Threat, Prioritize Threat] |A.Threat Level > 5 | |
Response[Mitigate Threat, Report Threat] | | |
Chain Succession[Report Threat, Review Threat] |A.Threat Status is Reported | |
Existence[Analyze Threat] | |
Chain Response[Mitigate Threat, Review Threat] | | |0,48,h
Not Chain Succession[Mitigate Threat, Mitigate Threat] | |A.Mitigation Strategy != T.Mitigation Strategy |
"""

In [ ]:
#Instanciating the ExtendedDeclareModel with the string of the MPDeclareModel.
declare_model = ExtendedDeclareModel().parse_from_string(model)

# Fetching activities, constraints, attributes, and binds from the model
model_activities = declare_model.get_model_activities()
model_attributes = declare_model.get_decl_attributes()
model_binds = declare_model.get_decl_binds()
model_constraints = declare_model.get_decl_model_constraints()
print("MP-Declare Model Constructs")
# Printing model activities
print("Model activities:")
print("-----------------")
for idx, act in enumerate(model_activities):
    print(idx, act)
print("\n")

# Printing model attributes
print("Model attributes:")
print("-----------------")
for idx, attr in enumerate(model_attributes):
    print(idx, attr)
print("\n")

# Printing model binds
print("Model binds:")
print("-------------")
for idx, bind in enumerate(model_binds):
    print(idx, bind)
print("\n")

# Printing model constraints
print("Model constraints:")
print("-----------------")
for idx, constr in enumerate(model_constraints):
    print(idx, constr)
print("\n")


## Step 2 — Intermediary description generation

The goal of this step is to generate intermediary descriptions of MP-Declare constructs. This is essential to have semantically enriched descriptions which can be used to evaluate if each part of the model is covered semantically and with correctness and completeness.

To achieve a consistent format of the intermediary descriptions we used some prompt engineering techniques: role-play prompting, knowledge infusion, few-shot learning and instruction tunning. Prompt engineering leverages large language models' capabilities without no need to fine-tune the model to achieve a goal.

In [ ]:
# The main prompt used to generate the intermediary descriptions.
decltotextassistant="""
<Assistant Prompt>
You are an an expert in declarative business process management, specifically for interpreting and describing Multi-Perspective Declare models (MP-Declare). Your primary task is to analyze the provided MP-Declare model and generate a comprehensive and consistent description. This description should meticulously reflect the structure and terminology of the model, maintaining exact naming and case sensitivity for model parts (activities, attributes, binds, and constraints).
Use the following knowledge base to know how to interpret the MP-Declare model:
<MP-Declare Model Specification>
#**Multi-Perspective Declare (MP-Declare) Constructs**

##**Activities**

<Activities Specification Guidelines>
Activities are specified with the word 'activity' followed by descriptive names, which should use active verbs, avoid abbreviations, and follow naming conventions (i.e., <Action Verb> <Object> or <Object> <Action Verb>).
</Activities Specification Guidelines>

**Examples of activities specification:**
<Activities Examples>
activity Driving Test
activity Initiate Purchase Order
</Activities Examples>
</Activities Specification Guidelines>

##**Binds**

<Binds Specification Guidelines>
Binds are specified as a link between activities and attributes. They have as a prefix the activity name followed by ':' and then a list of attributes, each separated by ','.

**Examples of binds specification:**
<Binds Examples>
Driving Test: Driver, Grade
Initiate Purchase Order: Purchase ID, Customer ID, Order Status, Order Value
</Binds Examples>
</Binds Specification Guidelines>

##**Attributes**
<Attributes Specification Guidelines>
MP-Declare models supports three types of attributes: integer, float and enumeration.

**Attribute Types:**
- **Integer:** Represents numeric integer values within a specified range.
- **Float:** Represents numeric values with floating-point within a specified range.
- **Enumeration:** Represents a limited set of possible values.

**Integer specification:**<Integer Attribute Specification Guidelines>[{Attribute Name}: integer between {min_value} and {max_value}], where {attribute name} is the name of attribute and {min_value} and {max_value} are two integer numbers.</Integer Attribute Specification Guideline>
**Float specification:**<Float Attribute Specification Guideline>[{Attribute Name}: float between {min_value} and {max_value}], where {attribute name} is the name of attribute and {min_value} and {max_value} are two float numbers.</Float Attribute Specification Guideline>
**Enumeration specification:**<Enumeration Specification Guideline>[{Attribute Name}: {value1}, {value2}, ..., {valuen}], where {attribute name} is the name of attribute and {value1} to {valuen} represent possible values for the attribute. These possible values should always be at least 2 and not infinite.</Enumeration Specification Guidelines>

**Examples of attributes specification:**
<Attributes Examples>
**Integer attribute example:** <Integer Attribute Example>Grade: integer between 1 and 5</Integer Attribute Example>.
**Float attribute example:** <Float Attribute Example>Order Value: float between 1.99 and 99999.9</Float Attribute Example>.
**Enumeration attribute example:** <Enumeration Attribute Example>Order Status: Approved, Not Approved, Shipped, Not Shipped</Enumeration Attribute Example>.
</Attributes Examples>
</Attributes Specification Guidelines>

##**Constraints**

MP-Declare has two types of templates: unary and binary. Following is specified the templates and their semantics.

###**Unary Templates**

Unary templates have only Activation activity.

<Unary Constraints Templates>
Constraint Template: Absence[n][Activation]
Semantics: Activation must not occur n times in the process.

Constraint Template: Exactly[n][Activation]
Semantics: Activation must occur exactly n times in the process.

Constraint Template: Existence[n][Activation]
Semantics: Activation must occur at least n times in the process.

Constraint Template: Init[Activation]
Semantics: Activation must be the first activity to occur in the process.

Constraint Template: End[Activation]
Semantics: Activation must be the last activity to occur in the process.
</Unary Constraints Templates>

###**Binary Templates**
Binary templates have both Activation and Target activities.
####**Binary Positive Templates**

<Binary Positive Templates>
Constraint Template: Alternate Precedence[Target, Activation]
Semantics: An Activation event can occur only if it was immediately preceded by a Target event, and no other Target event is allowed between that Target and the Activation. This ensures a one-to-one and ordered pairing between Targets and Activations, preventing any Activation from occurring without the necessary prior Target and avoiding overlapping or multiple Targets before an Activation.

Constraint Template: Alternate Response[Activation, Target]
Semantics: If an Activation occurs, then a Target must occur after that Activation and before any subsequent Activation occurs no other occurrence of Activation is allowed between an Activation and its corresponding Target. This ensures that each Activation is individually responded to by a Target in sequence.

Constraint Template: Alternate Succession[Activation, Target]
Semantics: Each occurrence of an Activation must eventually be followed by a Target, and each occurrence of a Target must be preceded by an Activation, with no intermediate occurrences of Activation or Target between them. This ensures a strict one-to-one and ordered pairing between Activations and Targets, preventing any overlapping or interruption by additional Activations or Targets in the sequence.

Constraint Template: Chain Precedence[Target, Activation]
Semantics: An Activation event can occur only if it is immediately preceded by a Target event, with no other events occurring in between. There can be multiple Activation-Target pairs within the process, each adhering to this immediate precedence rule. Activations may occur without being followed by a Target, but Targets cannot occur without such an immediate preceding Activation.

Constraint Template: Chain Response[Activation, Target]
Semantics: Whenever an Activation occurs, it must be immediately followed by a Target in the next event position, with no other events in between. This immediate succession ensures a direct and uninterrupted transition from Activation to Target. Multiple Activation-Target pairs can exist in a process instance, each adhering to this immediate succession rule. Events of Target can occur elsewhere in the process but not between an Activation and its corresponding Target.

Constraint Template: Chain Succession[Activation, Target]
Semantics: Whenever an Activation event occurs, it must be immediately followed by a Target event, with no other events in between. Conversely, whenever a Target event occurs, it must be immediately preceded by an Activation event, with no other events in between. This enforces a strict one-to-one and immediate pairing of Activations and Targets, ensuring that neither occurs without the other and that they occur consecutively without interruption.

Constraint Template: Co-Existence[Activation, Target]
Semantics: If Activation occurs in a process instance, then Target must also occur in the same process instance, and vice versa. This mutual obligation ensures that neither event can occur without the other. The events can occur multiple times and in any order relative to each other. If neither event occurs, the constraint is still considered satisfied.

Constraint Template: Precedence[Target, Activation]
Semantics: An Activation event is permitted to occur only if a Target event has occurred before it within the same process instance. Activations that occur without any prior Target are violations of the constraint. Multiple Targets can occur, and any number of events (including other Activations) can happen between a Target and a subsequent Activation. This ensures that certain actions cannot proceed without the necessary prerequisites being met first.

Constraint Template: Responded Existence[Activation, Target]
Semantics: If Activation occurs in a process instance, then Target must also occur at least once within the same process instance, either before or after the Activation. If Activation does not occur, there is no requirement for Target to occur. Multiple occurrences of Activation are satisfied by at least one occurrence of Target, and multiple Targets are allowed.

Constraint Template: Response[Activation, Target]
Semantics: If an Activation occurs, then a Target must eventually occur after the Activation at some point in the same process instance. There is no requirement on how soon or what events may occur in between. Multiple Activations can be pending at the same time, and a single Target can fulfill the obligation for all preceding Activations. If Activation does not occur, the constraint imposes no obligation on the occurrence of Target.

Constraint Template: Succession[Activation, Target]
Semantics: An Activation event must be followed by a Target event at some point later in the same process instance. Conversely, a Target event can occur only if an Activation event has occurred before it within the same process instance. This bidirectional and ordered dependency ensures that Activations impose obligations for future Targets, and Targets cannot occur without prior Activations. Multiple Activations and Targets are allowed, with each Activation eventually leading to a Target and each Target being justified by a preceding Activation..
</Binary Positive Templates>

####**Binary Negative Templates**

<Binary Negative Templates>
Constraint Template: Not Chain Succession[Activation, Target]
Semantics: An Activation event cannot be immediately followed by a Target event; there must be at least one other event between them. Similarly, a Target event cannot be immediately preceded by an Activation event. This constraint allows multiple occurrences of Activations and Targets in any order, provided they are not adjacent in the event sequence.

Constraint Template: Not Co-Existence[Activation, Target]
Semantics: The Activation event and the Target event cannot both occur within the same process instance. This means that if Activation occurs one or more times, Target must not occur at all in that process instance, and vice versa. It is acceptable for neither event to occur. The constraint enforces a mutual exclusion between Activation and Target, allowing for different process variants based on which event (if any) occurs.

Constraint Template: Not Succession[Activation, Target]
Semantics: An Activation event and a Target event must not occur such that the Activation is followed by the Target at any point later in the same process instance. Similarly, a Target event must not be preceded by an Activation event. This means that once an Activation occurs, Target cannot occur afterward, and if a Target occurs, it cannot have been preceded by an Activation. Both events can occur in the same process instance, but not in the prohibited sequence.

Constraint Template: Not Chain Precedence[Target, Activation]
Semantics: An Activation event cannot be directly preceded by a Target event, meaning that an Activation cannot occur immediately after a Target without any intervening events.

Constraint Template: Not Chain Response[Activation, Target]
Semantics: An Activation event cannot be directly followed by a Target event, meaning that a Target cannot occur immediately after an Activation without any intervening events.

Constraint Template: Not Precedence[Target, Activation]
Semantics: An Activation event cannot be preceded by a Target event within the same process instance. This means that if any Target event has occurred before an Activation event in the process instance, then that Activation is disallowed. Multiple Activations and Targets can occur, but all instances of the Activation must occur before any instance of the Target.

Constraint Template: Not Responded Existence[Activation, Target]
Semantics: If an Activation event occurs in a process instance, then a Target event must not occur anywhere in the same process instance. This means that the occurrence of Activation prohibits any occurrence of Target. However, if Activation does not occur, Target may occur freely. Multiple occurrences of Activation are allowed, provided Target does not occur at all in the process instance.

Constraint Template: Not Response[Activation, Target]
Semantics: An Activation event must not be followed by a Target event at any point later in the same process instance. This means that once an Activation occurs, Target cannot occur afterward. However, Target events occurring before the Activation are permitted. The constraint ensures that after the occurrence of Activation, the process does not allow any further occurrences of Target.
</Binary Negative Templates>

####**Choice Templates**

<Choice Templates>
Constraint Template: Choice[Activation, Target]
Semantics: At least one of the activities Activation or Target must occur in the process instance. This means that the process is compliant if Activation occurs, Target occurs, or both occur. There is no restriction on the order or the number of times they occur. However, if neither Activation nor Target occurs, the constraint is violated.

Constraint Template: Exclusive Choice[Activation, Target]
Semantics: Exactly one of the activities Activation or Target must occur in the process instance. This means that the process is compliant if only Activation occurs without any occurrence of Target, or only Target occurs without any occurrence of Activation. The process is non-compliant if both Activation and Target occur, or if neither occurs. The constraint enforces a mutual exclusivity and a mandatory occurrence of one of the specified activities.
</Choice Templates>

##**Conditions**
<Conditions>
There are three types of conditions in MP-Declare: activation, correlation and time.
<Activation and Correlation>
###**Activation and Correlation:**
<Activation Condition Example>
A.Phone Number
</Activation Condition Example>
<Correlation Condition Example>
T.Phone Number
</Correlation Condition Example>

###**Conditions Operations Guidelines**
<Conditions Operations Guidelines>
Conditions have operations on attributes and should always reference existing attributes values that are bound to the activities of the condition.

<Enumeration Operations>Operations on attribute values for enumeration: is, is not, in, not in.</Enumeration Operations>

**Examples of Operations in Enumeration Attributes:**
<Enumeration Operations Examples>
A.Transport Type is Car
A.Transport Type is not Car
A.Transport Type in (Car, Train)
A.Transport Type not in (Car, Train)

If both activation condition and correlation condition have the same attribute, == can be specified for equals, and != can be specified as different in the correlation condition section.

**Example for ==:**
<Example double equals operator>A.Transport Type == T.Transport Type</Example double equals operator>
**Example for or !=:**
<Example not equal to operator>A.Transport Type != T.Transport Type</Example not equal to operator>
</Enumeration Operations Examples>

<Integer or Float Operations>Operations on attribute values for numeric (integer or float) attribute: <, <=, =, =>, >.</Integer or Float Operations>
**Examples of Operations in Float or Integer Attributes:**
<Integer or Float Operations Examples>
A.Price > 10
A.Price <= 5
A.Price == 3
</Integer or Float Operations Examples>

<Logical Operations>Operations can be joined with 'and' and 'or'.</Logical Operations>
<Logical Operations Examples>
(T.Price <= 10) OR (T.Price > 100)
(T.Price = 10) AND (T.Transport Type is Car)
</Logical Operations Examples>

</Conditions Operations Guidelines>
</Activation and Correlation>

<Time Condition Guidelines>
The time condition must be specified as follows: min_value, max_value, measure. Where min_value and max_value are positive integers and min_value<=max_value and measure refers to the measure of time which can be s for seconds, m for minutes, h for hours or d for days.
</Time Condition Guidelines>

**Examples of Time Conditions Specification:**
<Time Condition Examples>
10,15,s: means that the condition must be triggered between 10 and 15 seconds.
5,13,m: means that the condition must be triggered between 5 and 13 minutes.
0,48,h: means that the condition must be triggered in at most 48 hours.
0,2,d: means that the condition must be triggered in at most 2 days.
</Time Condition Examples>
</Conditions>

Examples of model descriptions are provided below to guide your output and ensure alignment between data representation and text description. Your output will be the name of the construct and generated description.

###Activity Examples
####1
Name: activity Plan Event
Description:The 'Plan Event' activity represents the planning phase for organizing an event.
####2
Name: activity Collect Feedback
Description: The 'Collect Feedback' activity includes gathering feedback from guests and staff about the event.
####3
Name: activity Coordinate Photography
Description: The 'Coordinate Photography' activity involves managing photography services to capture moments during the event.

###Attributes Exampels
####1
Name: User ID: Integer between 1 and 1000
Description: The attribute 'User ID: Integer between 1 and 1000' represents the user's identification number for authentication purposes.
####2
Name: User Password: Float between 1000000 and 9000000
Description: The attribute 'User Password: Float between 1000000 and 9000000' represents the user's password for login.
####3
Name: Product Type: Cloths, radio, computer, notebook, food, chocolate
Description: The attribute 'Product Type: cloths, radio, computer, notebook, food, chocolate' specifies the types of products available for purchase.

###Binds Examples
####1
Name: bind Driving Test: Driver, Grade
Description: The bind 'Driving Test: Driver, Grade' specifies that the activity 'Driving Test' requires the attributes 'Driver' and 'Grade'.
####2
Name: bind Book Venue: Event ID, Venue
Description: The bind 'Book Venue: Event ID, Venue' links the 'Book Venue' activity with the attributes 'Event ID' and 'Venue'.
####3
Name: bind Hire Staff: Event ID, Staff Role
Description: The bind 'Hire Staff: Event ID, Staff Role' links the 'Hire Staff' activity with the attributes 'Event ID' and 'Staff Role'.

###Constraints Examples
####1
Name: Init[Plan Event] | |
Description: The 'Plan Event' must be the first activity to occur in the process.
####2
Name: End[Collect Feedback] | |
Description: The 'Collect Feedback' must be the last activity to occur in the process.
####3
Name: Existence[Conduct Internal Audit] | |
Description: The 'Conduct Internal Audit' activity must occur at least once in the process.
####4
Name: Existence4[Declare Contigency] | |
Description: The 'Declare Contigency' activity must occur at least 4 times in the process.
####5
Name: Absence[Stop Process] | |
Description: The 'Stop Process' must not occur in the process.
####6
Name: Absence3[Retry Access] | |
Description: The 'Retry Access' must not occur 3 times in the process.
####7:
Name: Exactly[Send Message] | |
Description: The 'Send Message' activity must occur exactly once in the process.
####8:
Name: Exactly5[Get Medicines] | |
Description: The 'Get Medicines' activity must occur exactly 5 times in the process.
####9:
Name: Alternate Precedence[Pay Bill, Get Goods] | | |
Description: The 'Get Goods' activity can occur only if it was immediately preceded by a 'Pay Bill' activity, and no other 'Pay Bill' event is allowed between the 'Pay Bill' and the 'Get Goods'.
####10:
Name: Alternate Response[Walk in the Park, Drink Water]
Description: If 'Walk in the Park' activity occurs, then 'Drink Water' activity  must occur after that 'Walk in the Park' event and before any subsequent 'Walk in the Park' occurs no other occurrence of 'Walk in the Park'is allowed between an 'Walk in the Park' and 'Drink Water'.
####11:
Name: Alternate Succession[Open Browser, Login] | | |
Description: Each occurrence of a 'Open Browser' activity must eventually be followed by a 'Login' activity, and each occurrence of a 'Login' must be preceded by 'Open Browser', with no intermediate occurrences of 'Open Browser' or 'Login' between them.
####12:
Name: Chain Precedence[Finish Process, Review Process] | | |
Description: The 'Review Process' activity can occur only if it is immediately preceded by 'Finish Process' activity, with no other activities occurring in between.
####13:
Name: Chain Response[Buy Medicine, Recover] | | |
Description: Whenever a 'Buy Medicine' activity occurs, it must be immediately followed by a 'Recover' activity in the next event position, with no other events in between. The 'Recover' activity is allowed to happen isolated.
####14:
Name: Chain Succession[Open the Door, Close the Door] | | |
Description: Whenever a 'Open the Door' activity occurs, it must be immediately followed by a 'Close the Door' activity, with no other events in between. Conversely, whenever a 'Close the Door' event occurs, it must be immediately preceded by a 'Open the Door' event, with no other events in between.
####15:
Name: Co-Existence[Send Inquiry, Execute Demand] | | |
Description: If 'Send Inquiry' activity occurs, then 'Execute Demand' must also occur in the same process instance, and vice-versa.
####16:
Name: Precedence[Serve Meals, Deliver Speech] | | |
Description: The 'Deliver Speech' activity is permitted to occur only if a 'Serve Meals' activity has occurred before it within the same process instance.
####17:
Name: Responded Existence[Login, Logout] | | |
Description: If 'Login' activity occurs in a process instance, then 'Logout' activity must also occur at least once within the same process instance, either before or after the 'Login' event. If 'Login' does not occur, there is no requirement for 'Logout' to occur. Multiple occurrences of 'Login' are satisfied by at least one occurrence of 'Logout', and multiple 'Logout' instances are allowed.
####18:
Name: Response[Open Browser, Fill URL] | | |
Description: If a 'Open Browser' activity occurs, then a 'Fill URL' activity must eventually occur after the 'Open Browser' event at some point in the same process instance. There is no requirement on how soon or what events may occur in between. Multiple 'Open Browser' can be pending at the same time, and a single 'Fill URL' can fulfill the obligation for all preceding 'Open Browser'. If 'Open Browser' does not occur, the constraint imposes no obligation on the occurrence of 'Fill URL'.
####19:
Name: Succession[Open Umbrella, Close Umbrella] | | |
Description: A 'Open Umbrella' activity must be followed by a 'Close Umbrella' activity at some point later in the same process instance. Conversely, a 'Close Umbrella' event can occur only if an 'Open Umbrella' event has occurred before it within the same process instance. This bidirectional and ordered dependency ensures that 'Open Umbrella' impose obligations for future 'Close Umbrella', and 'Close Umbrella' cannot occur without prior 'Open Umbrella'. Multiple 'Open Umbrella' and 'Close Umbrella' are allowed, with each 'Open Umbrella' eventually leading to a 'Close Umbrella' and each 'Close Umbrella' being justified by a preceding 'Open Umbrella'.
####20:
Name: Not Chain Succession[Style Hair, Take a Shower] | | |
Description: A 'Style Hair' activity cannot be immediately followed by a 'Take a Shower' activity; there must be at least one other event between them. Similarly, a 'Take a Shower' event cannot be immediately preceded by a 'Style Hair' event. This constraint allows multiple occurrences of 'Style Hair' and 'Take a Shower' in any order, provided they are not adjacent in the event sequence.
####21:
Name: Not Co-Existence[Go to Party, Stay at Home] | | |
Description: The 'Go to Party' activity and the 'Stay at Home' activity cannot both occur within the same process instance. This means that if 'Go to Party' event occurs one or more times, 'Stay at Home' must not occur at all in that process instance, and vice-versa. It is acceptable for neither event to occur. The constraint enforces a mutual exclusion between 'Go to Party' and 'Stay at Home', allowing for different process variants based on which event (if any) occurs.
####22:
Name: Not Succession[Eat Food, Run] | | |
Description: The 'Eat Food' activity and 'Run' activity must not occur such that the 'Eat Food' event is followed by the 'Run' event at any point later in the same process instance. Similarly, a 'Run' event must not be preceded by an 'Eat Food' event. This means that once an 'Eat Food' occurs, 'Run' cannot occur afterward, and if 'Run' event occurs, it cannot have been preceded by an 'Eat Food' event. Both events can occur in the same process instance, but not in the prohibited sequence.
####23:
Name: Not Chain Precedence[Order Confirmation, Order Placement] | | |
Description: The 'Order Placement' activity cannot be directly preceded by an 'Order Confirmation' activity, meaning that a 'Order Placement' event cannot occur immediately after an 'Order Confirmation' without any intervening events.
####24:
Name: Not Chain Response[Submit Order, Confirm Order] | | |
Description: The 'Submit Order' activity cannot be directly followed by a 'Confirm Order' activity, meaning that a 'Confirm Order' event cannot occur immediately after a 'Submit Order' without any intervening events.
####25:
Name: Not Precedence[Approve Order, Process Payment] | | |
Description: A 'Process Payment' activity cannot be preceded by a 'Approve Order' within the same process instance. If a 'Approve Order' event has occurred at any point before a 'Process Payment' event, then 'Process Payment' is disallowed. Multiple instances of 'Process Payment' and 'Approve Order' can occur, but all instances of 'Process Payment' must occur before any instance of 'Approve Order'.
####26:
Name: Not Responded Existence[Submit Order, Cancel Order] | | |
Description: If a 'Submit Order' activity occur in a process instance, then a 'Cancel Order' activity must not occur anywhere in the same process instance. This means that the occurrence of 'Submit Order' event prohibits any occurrence of 'Cancel Order' event. However, if 'Submit Order' does not occur, 'Cancel Order' may occur freely. Multiple occurrences of 'Submit Order' are allowed, provided 'Cancel Order' does not occur at all in the process instance.
####27:
Name: Not Response[Submit Application, Reject Application] | | |
Description: The 'Submit Application' activity must not be followed by a 'Reject Application' activity at any point later in the same process instance. This means that once a 'Submit Application' event occurs, 'Reject Application' cannot occur afterward. However, 'Reject Application' events occurring before the 'Submit Application' are permitted. The constraint ensures that after the occurrence of 'Submit Application', the process does not allow any further occurrences of 'Reject Application'.
####28:
Name: Choice[Receive Payment, Issue Refund] | | |
Description: At least one of the activities 'Receive Payment' or 'Issue Refund' must occur in the process instance. This means that the process is compliant if 'Receive Payment' event occurs, 'Issue Refund' event occurs, or both occur. There is no restriction on the order or the number of times they occur. However, if neither 'Receive Payment' nor 'Issue Refund' occurs, the constraint is violated.
####29:
Name: Exclusive Choice[Approve Application, Reject Application] | | |
Description: Exactly one of the activities 'Approve Application' or 'Reject Application' must occur in the process instance. This means that the process is compliant if only 'Approve Application' event occurs without any occurrence of 'Reject Application' event, or only 'Reject Application' occurs without any occurrence of 'Approve Application'. The process is non-compliant if both 'Approve Application' and 'Reject Application' occur, or if neither occurs. The constraint enforces a mutual exclusivity and a mandatory occurrence of one of the specified activities.
<\Assistant Prompt>"""

To structure each construct part (i.e., activities, attributes binds and constraints) description we employed function calling. This time, we separated each construct in a function call and applied a template of generation of description.

$$
\begin{array}{ll}
\textbf{Function} & \textbf{Description} \\
\hline
\text{get_activities_description} & f(\text{activities}) \rightarrow \text{description} \\
\text{get_attributes_description} & f(\text{attributes}) \rightarrow \text{description} \\
\text{get_binds_description} & f(\text{binds}) \rightarrow \text{description} \\
\text{get_constraints_description} & f(\text{constraints}) \rightarrow \text{description} \\
\hline
\end{array}
$$

$$
\begin{aligned}
\text{where} \\
f(x) &: \text{Apply template to each } x_i \in x \rightarrow \text{generated description} \\
x &\in \{\text{activities}, \text{attributes}, \text{binds}, \text{constraints}\} \\
x_i &: \{ \text{name} : \text{string}, \text{description} : \text{string} \} \\
\text{Required Properties:} \\
\text{name} &: \text{The name of the } x_i \\
\text{description} &: \text{The description of the } x_i \\
\end{aligned}
$$


In [ ]:
# 4 functions to get model description.
toolsdescription = [
  {
    "type": "function",
    "function": {
      "name": "get_activities_description",
      "description": "Fetch the activities of a given MP-Declare model and generate their descriptions following the template.",
      "parameters": {
        "type": "object",
        "properties": {
          "activities": {
            "type": "array",
            "description": "The activities contained in the model followed by generated descriptions.",
            "items": {
              "type": "object",
              "properties": {
                "name": {
                  "type": "string",
                  "description": "The name of the activity."
                },
                "description": {
                  "type": "string",
                  "description": "The description of the activity."
                }
              },
              "required": ["name", "description"]
            }
          }
        }
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "get_attributes_description",
      "description": "Fetch the attributes of a given MP-Declare model and generate their descriptions following the template.",
      "parameters": {
        "type": "object",
        "properties": {
          "attributes": {
            "type": "array",
            "description": "The attributes contained in the model followed by generated descriptions.",
            "items": {
              "type": "object",
              "properties": {
                "name": {
                  "type": "string",
                  "description": "The name of the attribute."
                },
                "description": {
                  "type": "string",
                  "description": "The description of the attribute."
                }
              },
              "required": ["name", "description"]
            }
          }
        }
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "get_binds_description",
      "description": "Fetch the binds of a given MP-Declare model and generate their descriptions following the template.",
      "parameters": {
        "type": "object",
        "properties": {
          "binds": {
            "type": "array",
            "description": "The binds contained in the model followed by generated descriptions.",
            "items": {
              "type": "object",
              "properties": {
                "name": {
                  "type": "string",
                  "description": "The name of the bind."
                },
                "description": {
                  "type": "string",
                  "description": "The description of the bind."
                }
              },
              "required": ["name", "description"]
            }
          }
        }
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "get_constraints_description",
      "description": "Fetch the constraints of a given MP-Declare model and generate their descriptions following the template.",
      "parameters": {
        "type": "object",
        "properties": {
          "constraints": {
            "type": "array",
            "description": "The constraints contained in the model followed by generated descriptions.",
            "items": {
              "type": "object",
              "properties": {
                "name": {
                  "type": "string",
                  "description": "The name of the constraint."
                },
                "description": {
                  "type": "string",
                  "description": "The description of the constraint."
                }
              },
              "required": ["name", "description"]
            }
          }
        }
      }
    }
  }
]

In [ ]:
model ="\n".join(declare_model.declare_model_lines)
intermediary_input_prompt = "New model to generate description of activities, attributes, binds and constraints based in the template and examples: \n" + model
print(intermediary_input_prompt)

In [ ]:
# Run prompt to generate intermediary description
messagesmptodesc=[
            {
                "role": "system",
                "content": decltotextassistant
            },
            {
                "role": "user",
                "content": intermediary_input_prompt
            }
        ]

set_step(step="generation.intermediary")
response_decltointermediary = chat_completion_request(
        messages=messagesmptodesc,
        tools= toolsdescription,
        tool_choice="auto",
        temperature=0
        )

In [ ]:
# The intermediary constructs come back as function/tool calls. If the model returns none (some
# providers lack native tool calling), fall back to a single structured Pydantic object of the four lists.
intermediary_tool_calls = response_decltointermediary.choices[0].message.tool_calls
if not intermediary_tool_calls:
    class _ConstructItem(BaseModel):
        name: str
        description: str
    class IntermediaryDescription(BaseModel):
        activities:  List[_ConstructItem] = Field(default_factory=list)
        attributes:  List[_ConstructItem] = Field(default_factory=list)
        binds:       List[_ConstructItem] = Field(default_factory=list)
        constraints: List[_ConstructItem] = Field(default_factory=list)
    set_step(step="generation.intermediary.fallback")
    _parsed = structured_output(messages=messagesmptodesc, response_format=IntermediaryDescription,
                                temperature=0).choices[0].message.parsed
    intermediary_tool_calls = [SimpleNamespace(function=SimpleNamespace(arguments=_parsed.model_dump_json()))]


In [ ]:
def process_intermediate_descriptions(data):
    model = {
        "Activities": [],
        "Attributes": [],
        "Binds": [],
        "Constraints": []
    }

    for item in data:
        func_args = json.loads(item.function.arguments)
        for key, value in func_args.items():
            bucket = model.get(key.capitalize())
            if bucket is None or not isinstance(value, list):
                continue
            for v in value:
                if isinstance(v, dict):
                    bucket.append({"name": v.get("name", ""), "description": v.get("description", "")})
                elif isinstance(v, str):
                    bucket.append({"name": v, "description": ""})

    model_string = ""
    for key, items in model.items():
        model_string += f"## {key}\n"
        for item in items:
            model_string += f"- **{item['name']}**: {item['description']}\n"
        model_string += "\n"

    return model_string, model


intermediary_description_string, intermediary_structured_model = process_intermediate_descriptions(intermediary_tool_calls)

## Step 3 — Final description (agentic synthesis)

This is the final step of our solution. It will take as input the intermediary descriptions and outputs the complete description of the MP-Declare model. It is necessary to mention that this entire description entangles all aspects of model constructs and how they interplay with each other.
In order to generate the description, the model will take a prompt with instructs in format of role-play and the intermediary descriptions of the process.

In [ ]:
guidelines="""
#Trace-Related:
    - Use Background for Init constraints to establish initial context.
    - Apply Conclusion for End constraint to close the model coherently.
    #Frequency:
    - Use Elaboration for Existence constraints to add depth and context regarding the presence of activities within the process.
    - Use Contrast for Absence constraints to highlight exclusions and reinforce process integrity by specifying what is not permissible.
    - Apply Summary for Exactly constraints to set fixed boundaries on activity occurrences, ensuring precision within the process model.
    #Dependency:
    - Use Cause-Effect for Responded Existence constraints to establish dependency, where one activity's occurrence necessitates the presence of another, enhancing coherence in the process.
    #Flexible Ordering:
    - Use Background for Precedence constraints to set up an initial context where one activity must occur before another, guiding the logical sequence within the model.
    - Apply Sequence for Response constraints to define an ordered flow where one activity follows another, ensuring a cohesive temporal structure in the model.
    - Use Cause-Effect for Succession constraints to establish a causal link, ensuring each activity follows logically from the previous one, creating a cohesive progression within the model.
    #Alternate Ordering:
    - Apply Condition-Consequence for Alternate Precedence constraints to ensure order consistency, requiring each occurrence of a target activity to be preceded by a specified activity without overlap, promoting balanced process flow.
    - Use Condition-Consequence for Alternate Response constraints to promote balance by ensuring each occurrence of an activation is followed by a unique target, with no intervening activations, supporting clear one-to-one activity pairing.
    - Use Condition-Consequence for Alternate Succession constraints to enforce an ordered structure with unique activity pairings, ensuring each activation is followed by a target and each target is preceded by an activation, without intervening occurrences, enhancing process coherence.
    #Immediate Ordering:
    - Use Sequence for Chain Precedence constraints to establish immediate dependency, ensuring that each target follows an activation directly, with no intervening events, making the sequence explicit and precise.
    - Apply Cause-Effect for Chain Response constraints to capture immediate succession, ensuring a direct cause-and-effect relationship where each activation is immediately followed by its target.
    - Apply Cause-Effect for Chain Succession constraints to enforce a tightly coupled sequence, ensuring each activation is directly succeeded by its target with no intervening activities.
    #Mutual Dependency:
    - Use Conjunction for Co-Existence constraints to establish mutual dependency, ensuring that both activities occur together within the process flow.
    #Exclusivity:
    - Use Contrast for Choice constraints to introduce alternative paths, allowing flexibility by permitting either activity to occur within the process.
    - Use Alternative for Exclusive Choice constraints to enforce mutually exclusive pathways, ensuring that exactly one of the specified activities occurs within the process.
    - Apply Contrast for Not Co-Existence constraints to set exclusion rules, ensuring that specified activities do not occur together within the same process instance, thereby maintaining process integrity by preventing conflicting events.
    #Negative Dependency:
    - Apply Contrast for Not Responded Existence constraints to prevent certain dependencies, ensuring that specified activities do not occur in response to one another, thus refining the model by disallowing specific contingent occurrences.
    #Negative Flexible Ordering:
    - Apply Contrast for Not Precedence constraints to restrict activities from occurring in a specific order, ensuring that designated activities do not precede others, thereby preserving the intended sequence within the model.
    - Apply Contrast for Not Response constraints to prevent activities from following a specific sequence, ensuring that designated activities do not occur after certain others, thus maintaining the integrity of the model's intended progression.
    - Apply Contrast for Not Succession constraints to restrict overlapping sequences, ensuring that certain activities do not follow others at any point, preserving the model's defined flow.
    #Negative Immediate Ordering:
    - Apply Contrast for Not Chain Precedence constraints to enforce non-immediate order restrictions, ensuring that specified activities are not immediately preceded by others, maintaining separation within the sequence.
    - Apply Contrast for Not Chain Response constraints to prevent immediate succession, ensuring that specified activities do not immediately follow one another, preserving structural integrity in the sequence.
    - Use Contrast for Not Chain Succession constraints to prevent direct succession, ensuring that specified activities do not immediately precede or follow each other, thereby refining the process model by excluding specific immediate sequences."""

response_template="""
The MP-Declare process model for the property transaction process is described as follows:

### **Trace-Related Constraints**
#### **Init Constraints**
The process begins with the activity 'Draft Contract,' which involves creating the initial version of the contract with 'Contract Version' and 'Contract Status' attributes.
#### **End Constraints**
The process concludes with the activity 'Close Deal,' which involves finalizing the property transaction and completing all necessary paperwork. This activity must be the last to occur in the process and must be completed within 30 days.

### **Frequency Constraints**

#### **Existence Constraint**
The 'Review Contract' activity must occur at least once in the process. This ensures that the contract is examined for accuracy and completeness, adding depth and reliability to the process.

### **Flexible Ordering Constraints**

#### **Precedence Constraints**
1. The activity 'Approve Contract' is permitted to occur only if a 'Conduct Title Search' activity has occurred before it. The 'Conduct Title Search' activity must verify that the 'Contract Status is Reviewed' and the 'Title Search Result is Clear,' ensuring that the contract is thoroughly reviewed and the property ownership is legally verified before approval.
2. The activity 'Negotiate Terms' is permitted to occur only if an 'Assess Property Value' activity has occurred before it, with the condition that the 'Property Value' is greater than 50,000. This ensures that the negotiation of terms is based on a valid and substantial property valuation.
3. The activity 'Close Deal' is permitted to occur only if a 'Verify Funds' activity has occurred before it. The 'Verify Funds' activity must confirm that the 'Deal Status is Open' and that the 'Funds Verified' are greater than or equal to the 'Property Value,' ensuring financial readiness before finalizing the deal.

#### **Response Constraints**
1. If the 'Draft Contract' activity occurs with the condition 'Contract Status is Draft,' then the 'Review Contract' activity must occur within 2 days. This ensures a timely review of the drafted contract, maintaining the process's momentum.
2. If the 'Sign Contract' activity occurs with the condition 'Contract Status is Signed,' then the 'Register Property' activity must occur with the condition 'Registration Status is Pending' within 5 days. This ensures that the property registration follows promptly after the contract is signed, maintaining the legal and procedural flow.

#### **Succession Constraints**
Whenever an 'Approve Contract' activity occurs with the condition 'Contract Status is Approved,' it must be immediately followed by a 'Sign Contract' activity with the condition 'Contract Status == Contract Status' within 1 day. This establishes a direct causal link between the approval and signing of the contract, ensuring a seamless progression in the process.

### **Exclusivity Constraints**

#### **Not Co-Existence Constraints**
The 'Sign Contract' activity and the 'Conduct Title Search' activity cannot both occur within the same process instance if the conditions 'Contract Status is Approved' and 'Title Search Result is Issues Found' are met. This exclusion ensures that a contract cannot be signed if the title search reveals issues, thereby maintaining the integrity of the process by preventing conflicting events.

### **Immediate Ordering Constraints**

#### **Chain Succession Constraints**
Whenever an 'Approve Contract' activity occurs with the condition 'Contract Status is Approved,' it must be immediately followed by a 'Sign Contract' activity with the condition 'Contract Status == Contract Status' within 1 day. This tightly coupled sequence ensures that the approval and signing of the contract occur in immediate succession, leaving no room for delays or intervening activities.
"""

In [ ]:
# Step 3 packaged as a function so the SAME agentic synthesis serves the single-model demo (below) and
# the batch (end of this section). The generator LLM and the intermediary descriptions are parameters.
def crew_narrative(intermediary_description_string, gen_model=DEFAULT_MODEL):
    narrative_llm = LLM(
        model=gen_model,
        temperature=0,
        seed=42,
        verbose=True,
        **_reasoning_body(gen_model)
    )
    narrative_agent = Agent(
        role="Narrative Generator",
        goal=("Generate an interleaved narrative of given process model's constraints."),
        backstory=("You are an expert in narrative generation, specializing in creating cohesive descriptions of MP-Declare process models's constraints using rhetorical relations from the Rethorical Structure Theory."),
        verbose=True,
        memory=True,
        llm=narrative_llm,
    )
    CreateNarrative = Task(
        description=(
            "The MP-Declare model you will use to generate the narrative: \n" + intermediary_description_string + "\n"
            "Use the following rhetorical relations for the given group of MP-Declare constraints: \n" + guidelines
        ),
        expected_output="The interleaved narrative of the given MP-Declare process model's constraints. Strictly follow the following response template: \n" + response_template,
        agent=narrative_agent
    )
    crew = Crew(
        agents=[narrative_agent],
        tasks=[CreateNarrative],
        process=Process.sequential,
        verbose=True
    )
    return crew.kickoff().raw

In [ ]:
# Run Step 3 on the demo model (the running example) with the default generator.
set_step(step="generation.narrative")
narrative_text = crew_narrative(intermediary_description_string, DEFAULT_MODEL)

### Assemble the final Nisaba description

In [ ]:
nisaba_description = f"#MP-Declare Model\n{intermediary_description_string}\n## Interleaved Description\n{narrative_text}"

In [ ]:
print(nisaba_description)

# 4. Evaluation

The generated Nisaba description and the zero-shot baseline are assessed with objective, generator-independent instruments, organised by the three quality dimensions: **correctness** (Semantic Distance over the reverse-generated model), **completeness** (complexity metrics), and **understandability** (readability, the RST relation metric, its DMRST neural corroboration, and comprehension QA). Reconstruction (reverse generation) is run first, since correctness compares the model reconstructed from each description against the original.

## Evaluation set (batch)

The batch complexity metrics run over the held-out **40-model `Evaluation_SoSyM`** set (generated with Terpsichora). Upload/extract `Evaluation_SoSyM` into Colab, then run the per-model pipeline for each `.decl` under `<Provider>/<FC|SO> Batch <n>/model<ID>/model<ID>.decl`. The single-model cells above illustrate the pipeline on one demo model: the correctness and understandability metrics below run on that in-memory model, while the complexity metrics iterate the batch.

In [ ]:
# === Evaluation set configuration ===
EVAL_ROOT = "/content/Evaluation_SoSyM"   # upload/extract Evaluation_SoSyM here
SOSYM_MODEL_FILES = sorted(glob.glob(os.path.join(EVAL_ROOT, "*", "* Batch *", "model*", "*.decl")))
print(f"{len(SOSYM_MODEL_FILES)} models found under {EVAL_ROOT}")

# Two ways to run the pipeline:
#   - SINGLE MODEL (the walkthrough above): the running example is already in `model`; to use an
#     evaluation-set model instead, set  model = open(SOSYM_MODEL_FILES[i], encoding="utf-8").read()
#     and re-run Steps 1-3 + the metric cells below.
#   - BATCH (all models x all generators): the "## Batch" section at the end of this Evaluation part runs
#     the SAME pipeline over every model and generator, then evaluates and saves the results.
# Batch knobs (defaults = the paper's setup: all evaluation-set models x all four generators):
BATCH_OUT        = "/content/nisaba_batch"     # batch outputs mirror the source tree under here
BATCH_MODELS     = SOSYM_MODEL_FILES           # restrict e.g. SOSYM_MODEL_FILES[:1] for a quick trial
BATCH_GENERATORS = GENERATORS                  # restrict e.g. {"deepseek": GENERATORS["deepseek"]}

## Baseline (zero-shot generation)

In [ ]:
#Generate Zero-shot description
messageszeroshot=[
            {
                "role": "user",
                "content":"Generate a natural language description of the following Multi-Perspective Declare model:\n" + model
            }
        ]

set_step(step="zeroshot")
response_zeroshotdescription = chat_completion_request(
        messages=messageszeroshot,
        temperature=0
        )
zeroshot_description=response_zeroshotdescription.choices[0].message.content

In [ ]:
#Save descriptions
with open("nisaba_description.txt", "w") as file:
    file.write(nisaba_description)

with open("zero-shot_description.txt", "w") as file:
    file.write(zeroshot_description)


## Reconstruction (reverse generation)

In [ ]:
# The main prompt used to generate the model reconstruction from descriptions.
decl_reconstruction_assistant="""
<Assistant Prompt>
You are an an expert in declarative business process management, specifically for interpreting and modeling Multi-Perspective Declare models (MP-Declare). Your primary task is to analyze the provided MP-Declare description and model the process in the given schema:
Use the following knowledge of MP-Declare to know it syntax and how to model:
<MP-Declare Model Specification>
#**Multi-Perspective Declare (MP-Declare) Constructs**

##**Activities**

<Activities Specification Guidelines>
Activities are specified with the word 'activity' followed by descriptive names, which should use active verbs, avoid abbreviations, and follow naming conventions (i.e., <Action Verb> <Object> or <Object> <Action Verb>).
</Activities Specification Guidelines>

**Examples of activities specification:**
<Activities Examples>
activity Driving Test
activity Initiate Purchase Order
</Activities Examples>
</Activities Specification Guidelines>

##**Binds**

<Binds Specification Guidelines>
Binds are specified as a link between activities and attributes. They have as a prefix the activity name followed by ':' and then a list of attributes, each separated by ','.

**Examples of binds specification:**
<Binds Examples>
Driving Test: Driver, Grade
Initiate Purchase Order: Purchase ID, Customer ID, Order Status, Order Value
</Binds Examples>
</Binds Specification Guidelines>

##**Attributes**
<Attributes Specification Guidelines>
MP-Declare models supports three types of attributes: integer, float and enumeration.

**Attribute Types:**
- **Integer:** Represents numeric integer values within a specified range.
- **Float:** Represents numeric values with floating-point within a specified range.
- **Enumeration:** Represents a limited set of possible values.

**Integer specification:**<Integer Attribute Specification Guidelines>[{Attribute Name}: integer between {min_value} and {max_value}], where {attribute name} is the name of attribute and {min_value} and {max_value} are two integer numbers.</Integer Attribute Specification Guideline>
**Float specification:**<Float Attribute Specification Guideline>[{Attribute Name}: float between {min_value} and {max_value}], where {attribute name} is the name of attribute and {min_value} and {max_value} are two float numbers.</Float Attribute Specification Guideline>
**Enumeration specification:**<Enumeration Specification Guideline>[{Attribute Name}: {value1}, {value2}, ..., {valuen}], where {attribute name} is the name of attribute and {value1} to {valuen} represent possible values for the attribute. These possible values should always be at least 2 and not infinite.</Enumeration Specification Guidelines>

**Examples of attributes specification:**
<Attributes Examples>
**Integer attribute example:** <Integer Attribute Example>Grade: integer between 1 and 5</Integer Attribute Example>.
**Float attribute example:** <Float Attribute Example>Order Value: float between 1.99 and 99999.9</Float Attribute Example>.
**Enumeration attribute example:** <Enumeration Attribute Example>Order Status: Approved, Not Approved, Shipped, Not Shipped</Enumeration Attribute Example>.
</Attributes Examples>
</Attributes Specification Guidelines>

##**Constraints**

MP-Declare has two types of templates: unary and binary. Following is specified the templates and their semantics.

###**Unary Templates**

Unary templates have only Activation activity.

<Unary Constraints Templates>
Constraint Template: Absence[n][Activation]
Semantics: Activation must not occur n times in the process.

Constraint Template: Exactly[n][Activation]
Semantics: Activation must occur exactly n times in the process.

Constraint Template: Existence[n][Activation]
Semantics: Activation must occur at least n times in the process.

Constraint Template: Init[Activation]
Semantics: Activation must be the first activity to occur in the process.

Constraint Template: End[Activation]
Semantics: Activation must be the last activity to occur in the process.
</Unary Constraints Templates>

###**Binary Templates**
Binary templates have both Activation and Target activities.
####**Binary Positive Templates**

<Binary Positive Templates>
Constraint Template: Alternate Precedence[Target, Activation]
Semantics: An Activation event can occur only if it was immediately preceded by a Target event, and no other Target event is allowed between that Target and the Activation. This ensures a one-to-one and ordered pairing between Targets and Activations, preventing any Activation from occurring without the necessary prior Target and avoiding overlapping or multiple Targets before an Activation.

Constraint Template: Alternate Response[Activation, Target]
Semantics: If an Activation occurs, then a Target must occur after that Activation and before any subsequent Activation occurs no other occurrence of Activation is allowed between an Activation and its corresponding Target. This ensures that each Activation is individually responded to by a Target in sequence.

Constraint Template: Alternate Succession[Activation, Target]
Semantics: Each occurrence of an Activation must eventually be followed by a Target, and each occurrence of a Target must be preceded by an Activation, with no intermediate occurrences of Activation or Target between them. This ensures a strict one-to-one and ordered pairing between Activations and Targets, preventing any overlapping or interruption by additional Activations or Targets in the sequence.

Constraint Template: Chain Precedence[Target, Activation]
Semantics: An Activation event can occur only if it is immediately preceded by a Target event, with no other events occurring in between. There can be multiple Activation-Target pairs within the process, each adhering to this immediate precedence rule. Activations may occur without being followed by a Target, but Targets cannot occur without such an immediate preceding Activation.

Constraint Template: Chain Response[Activation, Target]
Semantics: Whenever an Activation occurs, it must be immediately followed by a Target in the next event position, with no other events in between. This immediate succession ensures a direct and uninterrupted transition from Activation to Target. Multiple Activation-Target pairs can exist in a process instance, each adhering to this immediate succession rule. Events of Target can occur elsewhere in the process but not between an Activation and its corresponding Target.

Constraint Template: Chain Succession[Activation, Target]
Semantics: Whenever an Activation event occurs, it must be immediately followed by a Target event, with no other events in between. Conversely, whenever a Target event occurs, it must be immediately preceded by an Activation event, with no other events in between. This enforces a strict one-to-one and immediate pairing of Activations and Targets, ensuring that neither occurs without the other and that they occur consecutively without interruption.

Constraint Template: Co-Existence[Activation, Target]
Semantics: If Activation occurs in a process instance, then Target must also occur in the same process instance, and vice versa. This mutual obligation ensures that neither event can occur without the other. The events can occur multiple times and in any order relative to each other. If neither event occurs, the constraint is still considered satisfied.

Constraint Template: Precedence[Target, Activation]
Semantics: An Activation event is permitted to occur only if a Target event has occurred before it within the same process instance. Activations that occur without any prior Target are violations of the constraint. Multiple Targets can occur, and any number of events (including other Activations) can happen between a Target and a subsequent Activation. This ensures that certain actions cannot proceed without the necessary prerequisites being met first.

Constraint Template: Responded Existence[Activation, Target]
Semantics: If Activation occurs in a process instance, then Target must also occur at least once within the same process instance, either before or after the Activation. If Activation does not occur, there is no requirement for Target to occur. Multiple occurrences of Activation are satisfied by at least one occurrence of Target, and multiple Targets are allowed.

Constraint Template: Response[Activation, Target]
Semantics: If an Activation occurs, then a Target must eventually occur after the Activation at some point in the same process instance. There is no requirement on how soon or what events may occur in between. Multiple Activations can be pending at the same time, and a single Target can fulfill the obligation for all preceding Activations. If Activation does not occur, the constraint imposes no obligation on the occurrence of Target.

Constraint Template: Succession[Activation, Target]
Semantics: An Activation event must be followed by a Target event at some point later in the same process instance. Conversely, a Target event can occur only if an Activation event has occurred before it within the same process instance. This bidirectional and ordered dependency ensures that Activations impose obligations for future Targets, and Targets cannot occur without prior Activations. Multiple Activations and Targets are allowed, with each Activation eventually leading to a Target and each Target being justified by a preceding Activation..
</Binary Positive Templates>

####**Binary Negative Templates**

<Binary Negative Templates>
Constraint Template: Not Chain Succession[Activation, Target]
Semantics: An Activation event cannot be immediately followed by a Target event; there must be at least one other event between them. Similarly, a Target event cannot be immediately preceded by an Activation event. This constraint allows multiple occurrences of Activations and Targets in any order, provided they are not adjacent in the event sequence.

Constraint Template: Not Co-Existence[Activation, Target]
Semantics: The Activation event and the Target event cannot both occur within the same process instance. This means that if Activation occurs one or more times, Target must not occur at all in that process instance, and vice versa. It is acceptable for neither event to occur. The constraint enforces a mutual exclusion between Activation and Target, allowing for different process variants based on which event (if any) occurs.

Constraint Template: Not Succession[Activation, Target]
Semantics: An Activation event and a Target event must not occur such that the Activation is followed by the Target at any point later in the same process instance. Similarly, a Target event must not be preceded by an Activation event. This means that once an Activation occurs, Target cannot occur afterward, and if a Target occurs, it cannot have been preceded by an Activation. Both events can occur in the same process instance, but not in the prohibited sequence.

Constraint Template: Not Chain Precedence[Target, Activation]
Semantics: An Activation event cannot be directly preceded by a Target event, meaning that an Activation cannot occur immediately after a Target without any intervening events.

Constraint Template: Not Chain Response[Activation, Target]
Semantics: An Activation event cannot be directly followed by a Target event, meaning that a Target cannot occur immediately after an Activation without any intervening events.

Constraint Template: Not Precedence[Target, Activation]
Semantics: An Activation event cannot be preceded by a Target event within the same process instance. This means that if any Target event has occurred before an Activation event in the process instance, then that Activation is disallowed. Multiple Activations and Targets can occur, but all instances of the Activation must occur before any instance of the Target.

Constraint Template: Not Responded Existence[Activation, Target]
Semantics: If an Activation event occurs in a process instance, then a Target event must not occur anywhere in the same process instance. This means that the occurrence of Activation prohibits any occurrence of Target. However, if Activation does not occur, Target may occur freely. Multiple occurrences of Activation are allowed, provided Target does not occur at all in the process instance.

Constraint Template: Not Response[Activation, Target]
Semantics: An Activation event must not be followed by a Target event at any point later in the same process instance. This means that once an Activation occurs, Target cannot occur afterward. However, Target events occurring before the Activation are permitted. The constraint ensures that after the occurrence of Activation, the process does not allow any further occurrences of Target.
</Binary Negative Templates>

####**Choice Templates**

<Choice Templates>
Constraint Template: Choice[Activation, Target]
Semantics: At least one of the activities Activation or Target must occur in the process instance. This means that the process is compliant if Activation occurs, Target occurs, or both occur. There is no restriction on the order or the number of times they occur. However, if neither Activation nor Target occurs, the constraint is violated.

Constraint Template: Exclusive Choice[Activation, Target]
Semantics: Exactly one of the activities Activation or Target must occur in the process instance. This means that the process is compliant if only Activation occurs without any occurrence of Target, or only Target occurs without any occurrence of Activation. The process is non-compliant if both Activation and Target occur, or if neither occurs. The constraint enforces a mutual exclusivity and a mandatory occurrence of one of the specified activities.
</Choice Templates>

##**Conditions**
<Conditions>
There are three types of conditions in MP-Declare: activation, correlation and time.
<Activation and Correlation>
###**Activation and Correlation:**
<Activation Condition Example>
A.Phone Number
</Activation Condition Example>
<Correlation Condition Example>
T.Phone Number
</Correlation Condition Example>

###**Conditions Operations Guidelines**
<Conditions Operations Guidelines>
Conditions have operations on attributes and should always reference existing attributes values that are bound to the activities of the condition.

<Enumeration Operations>Operations on attribute values for enumeration: is, is not, in, not in.</Enumeration Operations>

**Examples of Operations in Enumeration Attributes:**
<Enumeration Operations Examples>
A.Transport Type is Car
A.Transport Type is not Car
A.Transport Type in (Car, Train)
A.Transport Type not in (Car, Train)

If both activation condition and correlation condition have the same attribute, == can be specified for equals, and != can be specified as different in the correlation condition section.

**Example for ==:**
<Example double equals operator>A.Transport Type == T.Transport Type</Example double equals operator>
**Example for or !=:**
<Example not equal to operator>A.Transport Type != T.Transport Type</Example not equal to operator>
</Enumeration Operations Examples>

<Integer or Float Operations>Operations on attribute values for numeric (integer or float) attribute: <, <=, =, =>, >.</Integer or Float Operations>
**Examples of Operations in Float or Integer Attributes:**
<Integer or Float Operations Examples>
A.Price > 10
A.Price <= 5
A.Price == 3
</Integer or Float Operations Examples>

<Logical Operations>Operations can be joined with 'and' and 'or'.</Logical Operations>
<Logical Operations Examples>
(T.Price <= 10) OR (T.Price > 100)
(T.Price = 10) AND (T.Transport Type is Car)
</Logical Operations Examples>

</Conditions Operations Guidelines>
</Activation and Correlation>

<Time Condition Guidelines>
The time condition must be specified as follows: min_value, max_value, measure. Where min_value and max_value are positive integers and min_value<=max_value and measure refers to the measure of time which can be s for seconds, m for minutes, h for hours or d for days.
</Time Condition Guidelines>

**Examples of Time Conditions Specification:**
<Time Condition Examples>
10,15,s: means that the condition must be triggered between 10 and 15 seconds.
5,13,m: means that the condition must be triggered between 5 and 13 minutes.
0,48,h: means that the condition must be triggered in at most 48 hours.
0,2,d: means that the condition must be triggered in at most 2 days.
</Time Condition Examples>
</Conditions>

Examples of model descriptions are provided below to guide your output and ensure alignment between data representation and text description. Your output will be the name of the construct and generated description.

###Activity Examples
####1
Name: activity Plan Event
Description:The 'Plan Event' activity represents the planning phase for organizing an event.
####2
Name: activity Collect Feedback
Description: The 'Collect Feedback' activity includes gathering feedback from guests and staff about the event.
####3
Name: activity Coordinate Photography
Description: The 'Coordinate Photography' activity involves managing photography services to capture moments during the event.

###Attributes Exampels
####1
Name: User ID: Integer between 1 and 1000
Description: The attribute 'User ID: Integer between 1 and 1000' represents the user's identification number for authentication purposes.
####2
Name: User Password: Float between 1000000 and 9000000
Description: The attribute 'User Password: Float between 1000000 and 9000000' represents the user's password for login.
####3
Name: Product Type: Cloths, radio, computer, notebook, food, chocolate
Description: The attribute 'Product Type: cloths, radio, computer, notebook, food, chocolate' specifies the types of products available for purchase.

###Binds Examples
####1
Name: bind Driving Test: Driver, Grade
Description: The bind 'Driving Test: Driver, Grade' specifies that the activity 'Driving Test' requires the attributes 'Driver' and 'Grade'.
####2
Name: bind Book Venue: Event ID, Venue
Description: The bind 'Book Venue: Event ID, Venue' links the 'Book Venue' activity with the attributes 'Event ID' and 'Venue'.
####3
Name: bind Hire Staff: Event ID, Staff Role
Description: The bind 'Hire Staff: Event ID, Staff Role' links the 'Hire Staff' activity with the attributes 'Event ID' and 'Staff Role'.

###Constraints Examples
####1
Name: Init[Plan Event] | |
Description: The 'Plan Event' must be the first activity to occur in the process.
####2
Name: End[Collect Feedback] | |
Description: The 'Collect Feedback' must be the last activity to occur in the process.
####3
Name: Existence[Conduct Internal Audit] | |
Description: The 'Conduct Internal Audit' activity must occur at least once in the process.
####4
Name: Existence4[Declare Contigency] | |
Description: The 'Declare Contigency' activity must occur at least 4 times in the process.
####5
Name: Absence[Stop Process] | |
Description: The 'Stop Process' must not occur in the process.
####6
Name: Absence3[Retry Access] | |
Description: The 'Retry Access' must not occur 3 times in the process.
####7:
Name: Exactly[Send Message] | |
Description: The 'Send Message' activity must occur exactly once in the process.
####8:
Name: Exactly5[Get Medicines] | |
Description: The 'Get Medicines' activity must occur exactly 5 times in the process.
####9:
Name: Alternate Precedence[Pay Bill, Get Goods] | | |
Description: The 'Get Goods' activity can occur only if it was immediately preceded by a 'Pay Bill' activity, and no other 'Pay Bill' event is allowed between the 'Pay Bill' and the 'Get Goods'.
####10:
Name: Alternate Response[Walk in the Park, Drink Water]
Description: If 'Walk in the Park' activity occurs, then 'Drink Water' activity  must occur after that 'Walk in the Park' event and before any subsequent 'Walk in the Park' occurs no other occurrence of 'Walk in the Park'is allowed between an 'Walk in the Park' and 'Drink Water'.
####11:
Name: Alternate Succession[Open Browser, Login] | | |
Description: Each occurrence of a 'Open Browser' activity must eventually be followed by a 'Login' activity, and each occurrence of a 'Login' must be preceded by 'Open Browser', with no intermediate occurrences of 'Open Browser' or 'Login' between them.
####12:
Name: Chain Precedence[Finish Process, Review Process] | | |
Description: The 'Review Process' activity can occur only if it is immediately preceded by 'Finish Process' activity, with no other activities occurring in between.
####13:
Name: Chain Response[Buy Medicine, Recover] | | |
Description: Whenever a 'Buy Medicine' activity occurs, it must be immediately followed by a 'Recover' activity in the next event position, with no other events in between. The 'Recover' activity is allowed to happen isolated.
####14:
Name: Chain Succession[Open the Door, Close the Door] | | |
Description: Whenever a 'Open the Door' activity occurs, it must be immediately followed by a 'Close the Door' activity, with no other events in between. Conversely, whenever a 'Close the Door' event occurs, it must be immediately preceded by a 'Open the Door' event, with no other events in between.
####15:
Name: Co-Existence[Send Inquiry, Execute Demand] | | |
Description: If 'Send Inquiry' activity occurs, then 'Execute Demand' must also occur in the same process instance, and vice-versa.
####16:
Name: Precedence[Serve Meals, Deliver Speech] | | |
Description: The 'Deliver Speech' activity is permitted to occur only if a 'Serve Meals' activity has occurred before it within the same process instance.
####17:
Name: Responded Existence[Login, Logout] | | |
Description: If 'Login' activity occurs in a process instance, then 'Logout' activity must also occur at least once within the same process instance, either before or after the 'Login' event. If 'Login' does not occur, there is no requirement for 'Logout' to occur. Multiple occurrences of 'Login' are satisfied by at least one occurrence of 'Logout', and multiple 'Logout' instances are allowed.
####18:
Name: Response[Open Browser, Fill URL] | | |
Description: If a 'Open Browser' activity occurs, then a 'Fill URL' activity must eventually occur after the 'Open Browser' event at some point in the same process instance. There is no requirement on how soon or what events may occur in between. Multiple 'Open Browser' can be pending at the same time, and a single 'Fill URL' can fulfill the obligation for all preceding 'Open Browser'. If 'Open Browser' does not occur, the constraint imposes no obligation on the occurrence of 'Fill URL'.
####19:
Name: Succession[Open Umbrella, Close Umbrella] | | |
Description: A 'Open Umbrella' activity must be followed by a 'Close Umbrella' activity at some point later in the same process instance. Conversely, a 'Close Umbrella' event can occur only if an 'Open Umbrella' event has occurred before it within the same process instance. This bidirectional and ordered dependency ensures that 'Open Umbrella' impose obligations for future 'Close Umbrella', and 'Close Umbrella' cannot occur without prior 'Open Umbrella'. Multiple 'Open Umbrella' and 'Close Umbrella' are allowed, with each 'Open Umbrella' eventually leading to a 'Close Umbrella' and each 'Close Umbrella' being justified by a preceding 'Open Umbrella'.
####20:
Name: Not Chain Succession[Style Hair, Take a Shower] | | |
Description: A 'Style Hair' activity cannot be immediately followed by a 'Take a Shower' activity; there must be at least one other event between them. Similarly, a 'Take a Shower' event cannot be immediately preceded by a 'Style Hair' event. This constraint allows multiple occurrences of 'Style Hair' and 'Take a Shower' in any order, provided they are not adjacent in the event sequence.
####21:
Name: Not Co-Existence[Go to Party, Stay at Home] | | |
Description: The 'Go to Party' activity and the 'Stay at Home' activity cannot both occur within the same process instance. This means that if 'Go to Party' event occurs one or more times, 'Stay at Home' must not occur at all in that process instance, and vice-versa. It is acceptable for neither event to occur. The constraint enforces a mutual exclusion between 'Go to Party' and 'Stay at Home', allowing for different process variants based on which event (if any) occurs.
####22:
Name: Not Succession[Eat Food, Run] | | |
Description: The 'Eat Food' activity and 'Run' activity must not occur such that the 'Eat Food' event is followed by the 'Run' event at any point later in the same process instance. Similarly, a 'Run' event must not be preceded by an 'Eat Food' event. This means that once an 'Eat Food' occurs, 'Run' cannot occur afterward, and if 'Run' event occurs, it cannot have been preceded by an 'Eat Food' event. Both events can occur in the same process instance, but not in the prohibited sequence.
####23:
Name: Not Chain Precedence[Order Confirmation, Order Placement] | | |
Description: The 'Order Placement' activity cannot be directly preceded by an 'Order Confirmation' activity, meaning that a 'Order Placement' event cannot occur immediately after an 'Order Confirmation' without any intervening events.
####24:
Name: Not Chain Response[Submit Order, Confirm Order] | | |
Description: The 'Submit Order' activity cannot be directly followed by a 'Confirm Order' activity, meaning that a 'Confirm Order' event cannot occur immediately after a 'Submit Order' without any intervening events.
####25:
Name: Not Precedence[Approve Order, Process Payment] | | |
Description: A 'Process Payment' activity cannot be preceded by a 'Approve Order' within the same process instance. If a 'Approve Order' event has occurred at any point before a 'Process Payment' event, then 'Process Payment' is disallowed. Multiple instances of 'Process Payment' and 'Approve Order' can occur, but all instances of 'Process Payment' must occur before any instance of 'Approve Order'.
####26:
Name: Not Responded Existence[Submit Order, Cancel Order] | | |
Description: If a 'Submit Order' activity occur in a process instance, then a 'Cancel Order' activity must not occur anywhere in the same process instance. This means that the occurrence of 'Submit Order' event prohibits any occurrence of 'Cancel Order' event. However, if 'Submit Order' does not occur, 'Cancel Order' may occur freely. Multiple occurrences of 'Submit Order' are allowed, provided 'Cancel Order' does not occur at all in the process instance.
####27:
Name: Not Response[Submit Application, Reject Application] | | |
Description: The 'Submit Application' activity must not be followed by a 'Reject Application' activity at any point later in the same process instance. This means that once a 'Submit Application' event occurs, 'Reject Application' cannot occur afterward. However, 'Reject Application' events occurring before the 'Submit Application' are permitted. The constraint ensures that after the occurrence of 'Submit Application', the process does not allow any further occurrences of 'Reject Application'.
####28:
Name: Choice[Receive Payment, Issue Refund] | | |
Description: At least one of the activities 'Receive Payment' or 'Issue Refund' must occur in the process instance. This means that the process is compliant if 'Receive Payment' event occurs, 'Issue Refund' event occurs, or both occur. There is no restriction on the order or the number of times they occur. However, if neither 'Receive Payment' nor 'Issue Refund' occurs, the constraint is violated.
####29:
Name: Exclusive Choice[Approve Application, Reject Application] | | |
Description: Exactly one of the activities 'Approve Application' or 'Reject Application' must occur in the process instance. This means that the process is compliant if only 'Approve Application' event occurs without any occurrence of 'Reject Application' event, or only 'Reject Application' occurs without any occurrence of 'Approve Application'. The process is non-compliant if both 'Approve Application' and 'Reject Application' occur, or if neither occurs. The constraint enforces a mutual exclusivity and a mandatory occurrence of one of the specified activities.
<\Assistant Prompt>"""

In [ ]:
# Reconstruction: recover an MP-Declare model from a natural-language description ALONE (the reader
# sees only the description -- not the source model or the generation guidelines). Defined here; applied
# below to the FINAL delivered description of each method, which is what makes the complexity /
# semantic-distance comparison meaningful (reconstructing from the structured intermediary would
# trivially preserve the model and hide the difference between methods).
def reconstruct_from_description(description):
    out = structured_output(
        messages=[
            {"role": "system", "content": decl_reconstruction_assistant},
            {"role": "user", "content": "Create a MP-Declare model of the following process using the "
                                        "knowledge from the assistant: \n " + description}],
        response_format=MPDeclareModel,
        temperature=0)
    return out.choices[0].message.parsed


In [ ]:
# Reconstruct BOTH final descriptions and save -> the discriminating comparison. Each description is
# reconstructed from the FINAL delivered text (Nisaba = intermediary + interleaved narrative; zero-shot
# = its prose), so the reconstructed model's complexity and Semantic Distance to the original measure
# what each *delivered* description actually preserves.
set_step(step="reconstruction")
for _tag, _desc in [("nisaba", nisaba_description), ("zero-shot", zeroshot_description)]:
    _m = reconstruct_from_description(_desc)
    with open(f"{_tag}_reconstructed.json", "w") as f:
        json.dump(_m.model_dump_json(), f, indent=4)
    with open(f"{_tag}_reconstructed.decl", "w") as f:
        f.write(_m.convert_to_string())
    print(f"reconstructed from {_tag} description -> {_tag}_reconstructed.decl")


## Correctness — Semantic Distance

In [ ]:
"""
MULTI-PERSPECTIVE semantic distance for MP-Declare models, built ON TOP of the
official declare4py parser (DeclareModel).

Motivation for the embedding: the round-trip
model->NL->model may introduce SYNONYMS/paraphrases (e.g., "Wood Cutting" ->
"Cut Wood"). Hence we align, via embedding + assignment (Jonker-Volgenant), ALL
the textual tokens where that makes sense:
  - ACTIVITY names       (act_map)
  - ATTRIBUTE names      (attr_map)
  - categorical VALUES   (val_map): enumeration members and the textual right-hand
                                    side of "is" conditions
Numbers (thresholds, ranges), operators, time windows and the TEMPLATE vocabulary
are compared EXACTLY (formal semantics must not be fuzzy).

Perspectives (each with its own Jaccard distance):
  control_flow    : (template, normalized-activities)
  temporal        : (constraint, time-window)
  data_conditions : (constraint, slot, role, normalized-attribute, op, normalized-value)
  binds           : (normalized-activity, normalized-attribute)
  attr_domains    : (normalized-attribute, canonical-domain)   [enum remapped via val_map]

Total = WEIGHTED mean of the ACTIVE perspectives (non-empty union in some model);
weights are customizable via DEFAULT_WEIGHTS or the `weights` argument. Perspectives
that are not applicable (empty in both models) are ignored (excluded from the mean).
"""
import re
import numpy as np
from scipy.optimize import linear_sum_assignment
from Declare4Py.ProcessModels.DeclareModel import DeclareModel


# ------------------------- embedding-based alignment -------------------------
def _label_stream():
    import string
    i = 0
    while True:
        n, s = i, ""
        while True:
            s = string.ascii_uppercase[n % 26] + s
            n = n // 26 - 1
            if n < 0:
                break
        yield s
        i += 1


def _l2(m):
    n = np.linalg.norm(m, axis=1, keepdims=True)
    n[n == 0] = 1.0
    return m / n


def semantic_pairing(list_A, list_B, embedding_model, tau=None, whiten=None):
    """Minimum-cost 1-to-1 pairing (cosine distance) via rectangular
    Jonker-Volgenant. Returns (pairs, unpaired_A, unpaired_B).

    knobs (default None = original behavior):
      tau    : REJECT pairs whose cost (1-cos) > tau -> they stay unpaired (avoids
               the spurious match forced by the bijection).
      whiten : 'center' subtracts the mean of {A union B} before normalizing
               (de-anisotropizes the cosine space)."""
    if not list_A or not list_B:
        return [], list(list_A), list(list_B)
    eA = np.asarray(embedding_model.encode(list_A), dtype=float)
    eB = np.asarray(embedding_model.encode(list_B), dtype=float)
    if whiten == 'center':
        mu = np.vstack([eA, eB]).mean(axis=0, keepdims=True)
        eA, eB = eA - mu, eB - mu
    eA, eB = _l2(eA), _l2(eB)
    cost = 1 - eA @ eB.T
    row, col = linear_sum_assignment(cost)
    pairs = []
    up_A_idx, up_B_idx = set(range(len(list_A))), set(range(len(list_B)))
    for i, j in zip(row, col):
        if tau is None or cost[i, j] <= tau:
            pairs.append((list_A[i], list_B[j], float(cost[i, j])))
            up_A_idx.discard(int(i)); up_B_idx.discard(int(j))
        # else: pair rejected by tau -> both remain unpaired
    up_A = [list_A[i] for i in sorted(up_A_idx)]
    up_B = [list_B[j] for j in sorted(up_B_idx)]
    return pairs, up_A, up_B


def build_mapping(list_A, list_B, embedding_model, tau=None, whiten=None):
    """N: paired -> common label; unpaired -> unique label. Takes the two token
    lists (one per model) and returns a dict token->label."""
    pairs, up_A, up_B = semantic_pairing(list_A, list_B, embedding_model, tau=tau, whiten=whiten)
    labels = _label_stream()
    m = {}
    for a, b, _ in pairs:
        lab = next(labels)
        m[a] = lab
        m[b] = lab
    for x in list(up_A) + list(up_B):
        m[x] = next(labels)
    return m


# ------------------------- loading via declare4py -------------------------
_OP = {'>': 'gt', '<': 'lt', '>=': 'ge', '<=': 'le', '=': 'eq', '==': 'eq',
       'is': 'eq', 'is not': 'ne', '!=': 'ne', 'in': 'in'}
_ATOM_RE = re.compile(r'^([ATB])\.(.+?)\s*(>=|<=|!=|==|=|>|<|is not|is|in)\s*(.+)$', re.I)


def _sanitize_decl(text):
    """Works around two declare4py fragilities on imperfect models:
      (1) binds WITHOUT an attribute ('bind X:' after strip) break split(': ');
      (2) domain lines ('Attr: value') whose attribute was never bound by a bind
          raise 'Unable to find attribute'.
    An empty bind carries no data; an orphan domain references a non-existent
    attribute -> both lines are dropped (no useful info for the metric)."""
    lines = text.split("\n")
    bound = set()
    for line in lines:
        s = line.strip()
        if s.startswith("bind ") and ":" in s:
            for a in s.split(":", 1)[1].split(","):
                a = a.strip()
                if a:
                    bound.add(a)
    out = []
    for line in lines:
        s = line.strip()
        if s.startswith("bind "):
            after = s.split(":", 1)[1].strip() if ":" in s else ""
            if after == "":
                continue                      # (1) empty bind
            out.append(line)
            continue
        if (":" in s and "[" not in s and not s.startswith("activity ")
                and DeclareModel.is_events_attrs_value_definition(s)):
            lhs = [a.strip() for a in s.split(":", 1)[0].split(",")]
            if any(a not in bound for a in lhs):
                continue                      # (2) orphan domain
            if "between" in s.lower():                      # (3) range domain w/o two numeric
                nums = re.findall(r"[-+]?(?:\d+\.?\d*|\.\d+)", s.split(":", 1)[1])
                if len(nums) < 2:                            # declare4py parse_attr_value -> IndexError
                    continue
        out.append(line)
    return "\n".join(out)


def _norm_val(v):
    """Normalize a condition value. Numeric -> float; otherwise lowercased string."""
    v = v.strip().strip('()').strip()
    try:
        return ('num', float(v))
    except ValueError:
        return ('str', v.lower())


def _parse_atoms(cond):
    """Split a condition into atoms (role, attr, op, ('num'|'str', value))."""
    if not cond:
        return []
    atoms = []
    for part in re.split(r'\s+(?:and|or)\s+', cond.strip(), flags=re.I):
        p = part.strip().strip('()').strip()
        if not p or p.lower() == 'true':
            continue
        m = _ATOM_RE.match(p)
        if m:
            role = m.group(1).upper()
            role = 'T' if role == 'B' else role
            attr = m.group(2).strip()
            op = _OP.get(m.group(3).lower().strip(), m.group(3).lower().strip())
            atoms.append((role, attr, op, _norm_val(m.group(4))))
        else:
            atoms.append(('?', p.lower(), '', ('str', p.lower())))
    return atoms


def _canonical_domain(attr):
    """Canonical domain. For enum, keep the raw (lowercased) members for later
    remapping via val_map; numbers stay exact."""
    av = getattr(attr, 'attr_value', None)
    if av is None:
        return ('none',)
    t = str(av.attribute_value_type)
    try:
        if t in ('integer_range', 'float_range'):
            return (t, float(av.value[0]), float(av.value[1]))
        if t == 'enumeration':
            return ('enum', frozenset(tok.get_name().strip().lower() for tok in av.value))
    except Exception:
        pass
    return (t, str(av.value_original).strip().lower())


def load_model(text):
    """Parse a .decl (string) via declare4py and extract the raw perspectives,
    also collecting the universes of attribute names and categorical values."""
    decl = text
    m = None
    for _ in range(30):
        decl = _sanitize_decl(decl)
        try:
            m = DeclareModel().parse_from_string(decl); break
        except ValueError as _e:                       # drop a bind whose event declare4py cannot match (hyphenated names)
            _mm = re.search(r"Unable to find the event or activity (.+)$", str(_e))
            if not _mm: raise
            _bad = _mm.group(1).strip()
            _kept = [l for l in decl.split("\n")
                     if not (l.strip().startswith("bind ") and l.strip()[4:].split(":", 1)[0].strip() == _bad)]
            if len(_kept) == len(decl.split("\n")): raise
            decl = "\n".join(_kept)
        except KeyError as _e:                          # dedupe a duplicated 'activity' declaration
            if "Multiple times the same event name" not in str(_e): raise
            _seen = set(); _out = []
            for l in decl.split("\n"):
                if l.strip().startswith("activity "):
                    _a = l.strip()[len("activity "):].strip()
                    if _a in _seen: continue
                    _seen.add(_a)
                _out.append(l)
            if len(_out) == len(decl.split("\n")): raise
            decl = "\n".join(_out)
    if m is None:
        m = DeclareModel().parse_from_string(_sanitize_decl(text))
    pm = m.parsed_model

    activities = list(m.activities)

    binds = {}
    for _et, evs in pm.events.items():
        for en, ev in evs.items():
            binds[en] = set(ev.attributes.keys())

    domains = {an: _canonical_domain(a) for an, a in pm.attributes_list.items()}

    constraints, cond_attr_names, value_tokens = [], set(), set()
    for _i, t in pm.templates.items():
        acts = [e.get_event_name() for e in t.events_activities if e is not None]
        slots = {}
        for slot, raw in (('act', t.get_activation_condition()),
                          ('tgt', t.get_target_condition())):
            atoms = _parse_atoms(raw)
            slots[slot] = atoms
            for (_role, attr, _op, val) in atoms:
                cond_attr_names.add(attr)
                if val[0] == 'str':
                    value_tokens.add(val[1])
        tm = t.get_time_condition()
        time = None
        if tm and tm.strip():
            parts = [x.strip() for x in tm.split(',')]
            try:
                time = (float(parts[0]), float(parts[1]), parts[2].lower())
            except Exception:
                time = tuple(parts)
        constraints.append({'template': t.get_template_name(), 'acts': acts,
                            'slots': slots, 'time': time})

    # categorical values: enumeration members + textual RHS of conditions
    for dom in domains.values():
        if dom and dom[0] == 'enum':
            value_tokens |= set(dom[1])

    attr_names = set(domains) | {a for s in binds.values() for a in s} | cond_attr_names
    return {'activities': activities, 'binds': binds, 'domains': domains,
            'constraints': constraints, 'attr_names': sorted(attr_names),
            'value_tokens': sorted(value_tokens)}


# ------------------------- perspectives + distance -------------------------
def _jaccard(s1, s2):
    if not s1 and not s2:
        return None            # perspective not applicable (empty in both)
    return 1 - len(s1 & s2) / len(s1 | s2)


def _perspective_sets(model, act_map, attr_map, val_map):
    A = lambda x: act_map.get(x, x)
    T = lambda x: attr_map.get(x, x)
    V = lambda x: val_map.get(x, x)

    def norm_value(val):
        kind, v = val
        return ('num', v) if kind == 'num' else ('str', V(v))

    control, temporal, data, binds, domains = set(), set(), set(), set(), set()

    for c in model['constraints']:
        cid = (c['template'], tuple(A(a) for a in c['acts']))
        control.add(cid)
        if c['time'] is not None:
            temporal.add((cid, c['time']))
        for slot, atoms in c['slots'].items():
            for (role, attr, op, val) in atoms:
                data.add((cid, slot, role, T(attr), op, norm_value(val)))

    for act, attrs in model['binds'].items():
        for at in attrs:
            binds.add((A(act), T(at)))

    for an, dom in model['domains'].items():
        if dom and dom[0] == 'enum':
            dom = ('enum', frozenset(V(v) for v in dom[1]))   # remap synonyms
        domains.add((T(an), dom))

    return {'control_flow': control, 'temporal': temporal,
            'data_conditions': data, 'binds': binds, 'attr_domains': domains}


DEFAULT_WEIGHTS = {'control_flow': 1.0, 'temporal': 1.0, 'data_conditions': 1.0,
                   'binds': 1.0, 'attr_domains': 1.0}

# Cost threshold (1-cos) adopted in the pairing: reject dissimilar matches instead
# of forcing them through the bijection (closes the blindness to unrelated relabels).
# Calibrated for gte-large (its cosine space is comparatively compressed).
# It does NOT affect identical/real pairs (cost 0 is never rejected). Use None to disable.
DEFAULT_TAU = 0.2


def compute_semantic_distance(decl_original, decl_reconstructed, embedding_model,
                              weights=None, tau=DEFAULT_TAU, whiten=None):
    """Multi-perspective distance. Returns a dict with 'total', 'perspectives', 'weights'.
    'total' is the WEIGHTED mean of the active perspectives. Weights are customizable:
    pass `weights` (dict perspective->weight) or edit DEFAULT_WEIGHTS. Default weights
    (1.0) => plain arithmetic mean.
    tau: cost threshold in the pairing (default DEFAULT_TAU=0.2; pass None to disable).
    whiten: 'center' de-anisotropizes the embedding space (see semantic_pairing)."""
    weights = weights or DEFAULT_WEIGHTS
    mo = load_model(decl_original)
    mr = load_model(decl_reconstructed)

    act_map = build_mapping(mo['activities'], mr['activities'], embedding_model, tau=tau, whiten=whiten)
    attr_map = build_mapping(mo['attr_names'], mr['attr_names'], embedding_model, tau=tau, whiten=whiten)
    val_map = build_mapping(mo['value_tokens'], mr['value_tokens'], embedding_model, tau=tau, whiten=whiten)

    so = _perspective_sets(mo, act_map, attr_map, val_map)
    sr = _perspective_sets(mr, act_map, attr_map, val_map)

    persp = {k: _jaccard(so[k], sr[k]) for k in so}

    active = {k: v for k, v in persp.items() if v is not None}
    if not active:
        total = 0.0
    else:
        wsum = sum(weights.get(k, 1.0) for k in active)
        total = sum(weights.get(k, 1.0) * v for k, v in active.items()) / wsum

    return {'total': total, 'perspectives': persp, 'weights': dict(weights)}


def compute_semantic_similarity(decl_original, decl_reconstructed, embedding_model):
    """Compat: returns only the total (float), drop-in for the previous metric."""
    return compute_semantic_distance(decl_original, decl_reconstructed, embedding_model)['total']


In [ ]:
# --- Semantic Distance (multi-perspective, gte-large, tau=EMB_TAU): original vs each reconstruction ---
# Closes the loop: the cell above reconstructed an MP-Declare model from each FINAL description; here we
# measure how far each reconstruction drifted from the ORIGINAL model (lower = the description preserved
# more of the model's multi-perspective semantics). 0.0 = full equivalence.
_orig_decl = "\n".join(declare_model.declare_model_lines)
_tau = EMB_TAU if 'EMB_TAU' in globals() else DEFAULT_TAU
for _tag in ["nisaba", "zero-shot"]:
    with open(f"{_tag}_reconstructed.decl", encoding="utf-8") as _f:
        _recon_decl = _f.read()
    _sd = compute_semantic_distance(_orig_decl, _recon_decl, embedding_model, tau=_tau)
    _persp = ", ".join(f"{k}={v:.2f}" for k, v in _sd['perspectives'].items() if v is not None)
    print(f"{_tag:9} semantic distance to original = {_sd['total']:.4f}   [{_persp}]")


## Completeness — Complexity metrics

In [ ]:
!unzip "/content/Evaluation_SoSyM.zip" -d /content/


In [ ]:
def calculate_size_metric(data):
    """
    Calculate the size metric for a process model.

    The size metric is defined as the sum of the number of activities and constraints in the model.

    Args:
        data (dict): A dictionary representing the process model data.

    Returns:
        int: The size metric, which is the sum of activities and constraints.
    """
    num_activities = len(data.get('activities', []))
    num_constraints = len(data.get('constraints', []))
    return num_activities + num_constraints

def calculate_density_metric(data):
    """
    Calculate the density metric for a process model.

    The density metric is defined as the ratio of constraints to activities within each connected component
    of the model graph. The function returns the maximum density across all components.

    Args:
        data (dict): A dictionary representing the process model data.

    Returns:
        float: The maximum density of any connected component in the model graph.
    """
    G = nx.Graph()

    # Add activities as nodes
    activities = data.get('activities', [])
    for activity in activities:
        G.add_node(activity['name'])

    # Add constraints as edges between activities
    constraints = data.get('constraints', [])
    for constraint in constraints:
        activation = constraint.get('activation')
        if isinstance(activation, dict):
            activation_name = activation.get('name')
        else:
            activation_name = None

        target = constraint.get('target')
        if isinstance(target, dict):
            target_name = target.get('name')
        else:
            target_name = None

        if activation_name and target_name:
            # Binary constraint
            G.add_edge(activation_name, target_name)
        elif activation_name:
            # Unary constraint
            # Optionally, add a self-loop to represent unary constraints
            G.add_edge(activation_name, activation_name)

    # Calculate the density for each connected component
    densities = []
    for component in nx.connected_components(G):
        subgraph = G.subgraph(component)
        num_constraints = subgraph.number_of_edges()
        num_activities = subgraph.number_of_nodes()
        if num_activities > 0:
            density = num_constraints / num_activities
            densities.append(density)

    # Return the maximum density across all components
    return max(densities) if densities else 0

def calculate_separability_metric(data):
    """
    Calculate the separability metric for a process model.

    The separability metric is defined as the ratio of the number of connected components to the total
    number of elements (activities + constraints) in the model.

    Args:
        data (dict): A dictionary representing the process model data.

    Returns:
        float: The separability metric, which is the ratio of connected components to total elements.
    """
    G = nx.Graph()

    # Add activities as nodes
    activities = data.get('activities', [])
    for activity in activities:
        G.add_node(activity['name'])

    # Add constraints as edges between activities
    constraints = data.get('constraints', [])
    for constraint in constraints:
        activation = constraint.get('activation')
        if isinstance(activation, dict):
            activation_name = activation.get('name')
        else:
            activation_name = None

        target = constraint.get('target')
        if isinstance(target, dict):
            target_name = target.get('name')
        else:
            target_name = None

        if activation_name and target_name:
            G.add_edge(activation_name, target_name)
        elif activation_name:
            G.add_node(activation_name)  # Ensure the node is in the graph

    # Calculate the number of connected components
    num_components = nx.number_connected_components(G)
    total_elements = len(activities) + len(constraints)

    if total_elements > 0:
        return num_components / total_elements
    else:
        return 0

def calculate_constraint_variability_metric(data):
    """
    Calculate the constraint variability metric for a process model.

    The constraint variability metric is defined as the maximum entropy of constraint types
    across the connected components of the model graph.

    Args:
        data (dict): A dictionary representing the process model data.

    Returns:
        float: The maximum entropy of constraint types across connected components, representing constraint variability.
    """
    G = nx.Graph()
    activities = data.get('activities', [])
    for activity in activities:
        G.add_node(activity['name'])

    constraints = data.get('constraints', [])
    for constraint in constraints:
        activation = constraint.get('activation')
        if isinstance(activation, dict):
            activation_name = activation.get('name')
        else:
            activation_name = None

        target = constraint.get('target')
        if isinstance(target, dict):
            target_name = target.get('name')
        else:
            target_name = None

        constraint_type = constraint.get('template')

        if activation_name and target_name:
            # Binary constraint
            G.add_edge(activation_name, target_name, type=constraint_type)
        elif activation_name:
            # Unary constraint
            G.add_edge(activation_name, activation_name, type=constraint_type)
        # If neither activation nor target has a name, skip this constraint

    # Get all constraint types
    all_constraint_types = set(nx.get_edge_attributes(G, 'type').values())
    num_constraint_types = len(all_constraint_types)

    if num_constraint_types == 0:
        return 0  # No constraints, so variability is 0

    max_entropy = 0

    # Calculate entropy for each connected component
    for component in nx.connected_components(G):
        subgraph = G.subgraph(component)
        component_constraints = list(nx.get_edge_attributes(subgraph, 'type').values())

        if component_constraints:
            count = Counter(component_constraints)
            total_constraints = len(component_constraints)

            if len(count) == 1:
                # Only one type of constraint, entropy is 0
                entropy = 0
            else:
                entropy = 0
                for constraint_type in all_constraint_types:
                    p = count[constraint_type] / total_constraints
                    if p > 0:
                        # Use log base 2 as an approximation
                        entropy -= p * math.log2(p) / math.log2(num_constraint_types)

            max_entropy = max(max_entropy, entropy)

    return max_entropy


def calculate_metrics(folder_path):
    """
    Calculate complexity metrics for each process model in a given folder and its subfolders.

    This function iterates over JSON files in a specified folder and its subfolders, calculates complexity metrics
    (size, density, separability, and constraint variability) for each process model, and stores the results
    in a pandas DataFrame.

    Args:
        folder_path (str): The path to the folder containing the JSON files.

    Returns:
        pd.DataFrame: A DataFrame containing the calculated metrics for each model.
    """
    metrics_results = []

    # Traverse the folder and subfolders
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.json'):
                file_path = os.path.join(root, file)
                with open(file_path, 'r') as f:
                    try:
                        content = f.read()
                        json_string = json.loads(content)
                        data = json.loads(json_string)
                    except json.JSONDecodeError as e:
                        print(f"Error decoding JSON from file {file_path}: {e}")
                        continue

                    # Calculate each metric
                    size_metric = calculate_size_metric(data)
                    density_metric = calculate_density_metric(data)
                    separability_metric = calculate_separability_metric(data)
                    constraint_variability_metric = calculate_constraint_variability_metric(data)

                    # Store the results for this model
                    metrics_results.append({
                        'File': file,
                        'Path': file_path,
                        'Size Metric': size_metric,
                        'Density Metric': density_metric,
                        'Separability Metric': separability_metric,
                        'Constraint Variability Metric': constraint_variability_metric
                    })

    # Convert results to a DataFrame for easy viewing
    df = pd.DataFrame(metrics_results)
    return df


def save_metrics_to_csv(metrics_df, folder_path, filename='complexity_metrics.csv'):
    """
    Save complexity metrics to a CSV file.

    This function takes a DataFrame containing complexity metrics, saves it to a specified folder as a CSV file,
    and ensures the file is named appropriately.

    Args:
        metrics_df (pd.DataFrame): The DataFrame containing the metrics to save.
        folder_path (str): The folder path where the CSV file will be saved.
        filename (str): The name of the CSV file (default: 'complexity_metrics.csv').

    Returns:
        str: The full path of the saved CSV file.
    """
    # Ensure the folder exists
    os.makedirs(folder_path, exist_ok=True)

    # Construct the full path for the CSV file
    file_path = os.path.join(folder_path, filename)

    # Save the DataFrame to CSV
    metrics_df.to_csv(file_path, index=False)

    return file_path


In [ ]:
df = calculate_metrics(EVAL_ROOT)


In [ ]:
save_metrics_to_csv(df, '/content/')

## Understandability

### Readability (FRE)

In [ ]:
# Readability: Flesch Reading Ease (FRE) via textstat.
def _md_to_text(s):
    s = re.sub(r"[`*#>_\[\]()]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def process_folder(folder_path, output_csv):
    """Flesch Reading Ease of every *_description.txt under folder_path (higher = easier to read)."""
    rows = []
    for root, _, files in os.walk(folder_path):
        for fn in files:
            if fn.endswith(".txt") and "description" in fn:
                text = _md_to_text(open(os.path.join(root, fn), encoding="utf-8").read())
                rows.append({"filename": fn, "flesch_reading_ease": round(textstat.flesch_reading_ease(text), 2)})
    df = pd.DataFrame(rows).sort_values("filename")
    df.to_csv(output_csv, index=False)
    return df

# Flesch Reading Ease of the delivered descriptions saved above (higher = easier to read).
readability_df = process_folder(".", "readability_scores.csv")
print(readability_df.to_string(index=False))


### RST relation metric (rhetorical-structure fidelity)

This objective metric measures whether a description realises the rhetorical relations the generation guidelines prescribe. The guidelines map each MP-Declare constraint **type** to a specific rhetorical relation, so a model deterministically defines a target relation profile. The cell below checks, per constraint, whether the description realises that relation using a fixed lexicon of discourse markers (no LLM). It is generator-independent and is corroborated by a neural RST discourse parser (DMRST, below).

In [ ]:
# ==========================================================================
# RST Relation Metric -- objective, generator-independent measure of rhetorical-structure
# fidelity. Guidelines map each constraint TYPE -> a prescribed RST relation;
# we check per constraint whether the description realises it via a fixed discourse-
# marker lexicon (standard PDTB/RST connectives, not tuned to Nisaba). No LLM.
# Runs on the in-memory nisaba_description vs zeroshot_description for the current model.
# ==========================================================================

RST_CONSTRAINT_RELATION = {
    "Init":"Background","End":"Conclusion","Existence":"Elaboration","Absence":"Contrast",
    "Exactly":"Summary","Responded Existence":"Cause-Effect","Precedence":"Background",
    "Response":"Sequence","Succession":"Cause-Effect","Alternate Precedence":"Condition-Consequence",
    "Alternate Response":"Condition-Consequence","Alternate Succession":"Condition-Consequence",
    "Chain Precedence":"Sequence","Chain Response":"Cause-Effect","Chain Succession":"Cause-Effect",
    "Co-Existence":"Conjunction","Choice":"Contrast","Exclusive Choice":"Alternative",
    "Not Co-Existence":"Contrast","Not Responded Existence":"Contrast","Not Precedence":"Contrast",
    "Not Response":"Contrast","Not Succession":"Contrast","Not Chain Precedence":"Contrast",
    "Not Chain Response":"Contrast","Not Chain Succession":"Contrast"}
RST_MARKERS = {
 "Background":[r"begins? with",r"\binitially\b",r"initial context",r"at the outset",r"\bbefore\b",r"prior to",r"\bearlier\b",r"\balready\b",r"sets? up",r"has (?:occurred|happened) before",r"must have (?:occurred|happened)"],
 "Conclusion":[r"\bfinally\b",r"\blastly\b",r"conclude[sd]?",r"in conclusion",r"to close",r"ends? with",r"must be the last",r"closes the"],
 "Elaboration":[r"\bspecifically\b",r"in particular",r"\bnotably\b",r"in detail",r"\bfurthermore\b",r"\bmoreover\b",r"\badditionally\b",r"adds? depth",r"at least once",r"\bnamely\b"],
 "Contrast":[r"\bhowever\b",r"in contrast",r"on the other hand",r"\bconversely\b",r"\bwhereas\b",r"\bbut\b",r"\byet\b",r"\balthough\b",r"\bthough\b",r"\bnevertheless\b",r"\bnonetheless\b",r"\binstead\b"],
 "Summary":[r"\bexactly\b",r"\bprecisely\b",r"in total",r"a total of",r"a fixed number",r"no more than",r"no fewer than",r"in summary",r"to summari[sz]e"],
 "Cause-Effect":[r"\bbecause\b",r"\btherefore\b",r"\bthus\b",r"\bhence\b",r"\bconsequently\b",r"as a result",r"so that",r"cause[- ]and[- ]effect",r"\bcausal\b",r"\btriggers?\b",r"leads? to",r"results? in",r"\bensuring\b",r"\bthereby\b"],
 "Sequence":[r"\bthen\b",r"\bafter\b",r"\bafterwards\b",r"\bsubsequently\b",r"\bnext\b",r"\bfollowing\b",r"\bonce\b",r"\beventually\b",r"\blater\b",r"is (?:immediately )?followed by",r"\bfollows\b",r"must (?:eventually )?occur after"],
 "Condition-Consequence":[r"\bif\b",r"\bwhen\b",r"\bwhenever\b",r"provided that",r"in case",r"only if",r"as long as",r"\bunless\b",r"on the condition"],
 "Conjunction":[r"both .* and",r"\btogether\b",r"co[- ]exist",r"\bjointly\b",r"as well as",r"in conjunction",r"must occur together"],
 "Alternative":[r"either .* or",r"\bexclusively\b",r"\balternatively\b",r"mutually exclusive",r"exactly one",r"one of the",r"or else"]}
_RST_C = {rel:[re.compile(p,re.I) for p in ps] for rel,ps in RST_MARKERS.items()}
_RST_TYPES = sorted(RST_CONSTRAINT_RELATION, key=len, reverse=True)
_RST_STOP = {"the","a","an","of","to","and","or","for"}
def _rst_stem(t):
    t=re.sub(r"[^a-z]","",t.lower())
    for s in ("ing","ed","es","s","e"):
        if len(t)>len(s)+2 and t.endswith(s): return t[:-len(s)]
    return t
def _rst_stems(x): return {_rst_stem(t) for t in re.findall(r"[A-Za-z]+",x) if t.lower() not in _RST_STOP}
def _rst_constraints(decl):
    out=[]
    for line in decl.splitlines():
        m=re.match(r"^\s*([A-Za-z][A-Za-z \-]*?)\s*\[(.*?)\]",line)
        if not m: continue
        head=m.group(1).strip()
        if head.lower() in ("activity","bind"): continue
        ct=next((t for t in _RST_TYPES if head.lower()==t.lower()),None) or next((t for t in _RST_TYPES if head.lower().startswith(t.lower())),None)
        if ct: out.append((ct, RST_CONSTRAINT_RELATION[ct], [a.strip() for a in m.group(2).split(",") if a.strip()]))
    return out
def _rst_sents(text):
    lines=[re.sub(r"^[-*0-9.\)]+\s*","",l.strip()).replace("**","") for l in text.splitlines() if l.strip() and not l.strip().startswith("#")]
    j=re.sub(r"(\d)\.(\d)",r"\1<DOT>\2"," ".join(lines))
    return [p.replace("<DOT>",".").strip() for p in re.split(r"(?<=[.!?;:])\s+",j) if p.strip()]
def _rst_mentions(sents, acts):
    toks=[{_rst_stem(t) for t in re.findall(r"[A-Za-z]+",a) if t.lower() not in _RST_STOP} for a in acts]
    return [s for s in sents if any(tk and tk<=_rst_stems(s) for tk in toks)]
def rst_relation_metric(text, constraints):
    sents=_rst_sents(text); mentioned=aligned=0; per=defaultdict(lambda:[0,0])
    for ct,rel,acts in constraints:
        cand=_rst_mentions(sents,acts)
        if not cand: continue
        mentioned+=1; per[ct][0]+=1
        if any(rx.search(" ".join(cand)) for rx in _RST_C[rel]): aligned+=1; per[ct][1]+=1
    return {"n":len(constraints),"coverage":mentioned/len(constraints) if constraints else 0.0,
            "aligned_rate":aligned/mentioned if mentioned else 0.0,"per_type":{k:tuple(v) for k,v in per.items()}}

# --- run on the current model (declare_model is the parsed ExtendedDeclareModel) ---
_decl = "\n".join(declare_model.declare_model_lines)
_cons = _rst_constraints(_decl)
_i = nisaba_description.find("## Interleaved Description")
_nis_narr = nisaba_description[_i:] if _i >= 0 else nisaba_description
_rn = rst_relation_metric(_nis_narr, _cons)
_rz = rst_relation_metric(zeroshot_description, _cons)
print("Target relation profile:", dict(Counter(r for _,r,_ in _cons)))
print(f"Nisaba    aligned_rate={_rn['aligned_rate']:.2f}  coverage={_rn['coverage']:.2f}")
print(f"Zero-shot aligned_rate={_rz['aligned_rate']:.2f}  coverage={_rz['coverage']:.2f}")
print("\nPer constraint type (prescribed relation realised, aligned/mentioned):")
for ct in sorted(set(_rn['per_type']) | set(_rz['per_type'])):
    n=_rn['per_type'].get(ct,(0,0)); z=_rz['per_type'].get(ct,(0,0))
    print(f"  {ct:22} {RST_CONSTRAINT_RELATION[ct]:22} Nisaba {n[1]}/{n[0]}   zero-shot {z[1]}/{z[0]}")


### DMRST neural RST-parser corroboration

As a robustness check on the lexical RST metric above, we run an independent **neural** discourse parser (DMRST; xlm-roberta-base backbone, GUM/RST-DT relation set) and map its relation labels onto the same Nisaba relation inventory. The two methods agree on the magnitude and the location of the Nisaba-vs-zero-shot gap, which is concentrated in the non-temporal relations (Contrast, Condition-Consequence, Cause-Effect). CPU-only. The ~1.26 GB checkpoint ships with the parser repo.

In [ ]:
# === DMRST neural RST-parser corroboration of the lexicon RST probe (CPU-safe, Colab) ===
# Independent neural discourse parser (DMRST; Liu/Shi/Chen 2021; xlm-roberta-base backbone) corroborating
# the lexicon RST metric above: confirms the Nisaba-vs-zero-shot NON-TEMPORAL relation gap
# (Contrast / Condition-Consequence / Cause-Effect).
import os, re, sys, json, csv, glob, subprocess
from collections import Counter

# --- 0. deps: transformers + gdown are pinned in the Requirements section ---
import torch
from transformers import AutoTokenizer, AutoModel

# --- 1. clone the parser (model code is device-agnostic; only MUL_main_*.py call .cuda) ---
REPO = "DMRST_Parser"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/seq-to-mind/DMRST_Parser.git", REPO], check=True)
sys.path.insert(0, REPO)

# --- 2. obtain the ~1.26 GB checkpoint -> depth_mode/Savings/ (ships with the repo; guard for missing) ---
CKPT = os.path.join(REPO, "depth_mode", "Savings", "multi_all_checkpoint.torchsave")
if not os.path.exists(CKPT):
    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    import gdown
    gdown.download(id="12Gc6mC6Qh0R_N_U60mx2jDQqRfuwQzZh", output=CKPT, quiet=False)
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 5e8:
    raise SystemExit(
        "DMRST checkpoint missing/incomplete (~1.26 GB expected). Download manually from "
        "https://drive.google.com/file/d/12Gc6mC6Qh0R_N_U60mx2jDQqRfuwQzZh/view -> " + CKPT)

from model_depth import ParsingNet  # pulls config.py / module.py / DataHandler.py from REPO

# --- 3. build + load on CPU. CPU 'patch' = never call .cuda() + map_location="cpu" ---
tok = AutoTokenizer.from_pretrained("xlm-roberta-base", use_fast=True)
bert = AutoModel.from_pretrained("xlm-roberta-base")          # ~1.1 GB frozen backbone
for p in bert.parameters():
    p.requires_grad = False
net = ParsingNet(bert, bert_tokenizer=tok)                   # CPU only (no .cuda())
try:
    state = torch.load(CKPT, map_location="cpu")
except Exception:
    state = torch.load(CKPT, map_location="cpu", weights_only=False)  # torch>=2.6
net.load_state_dict(state, strict=False)                    # only position_ids buffers differ
net.eval()

# --- 4. DMRST(RST-DT/GUM) relation -> Nisaba 10-relation set ---
DM2NIS = {"Background": "Background", "Elaboration": "Elaboration", "Contrast": "Contrast",
          "Comparison": "Contrast", "Summary": "Summary", "Cause": "Cause-Effect",
          "Explanation": "Cause-Effect", "Enablement": "Cause-Effect", "Temporal": "Sequence",
          "Condition": "Condition-Consequence", "Joint": "Conjunction"}   # rest -> "Other"
NONTEMPORAL = {"Contrast", "Condition-Consequence", "Cause-Effect"}

def parse_text(text):
    toks = tok.tokenize(text)                                # whole doc; DMRST self-segments EDUs
    if len(toks) < 2:
        return [], ""
    with torch.no_grad():
        _, _, spans, _, edu = net.TestingLoss([toks], input_EDU_breaks=None, LabelIndex=None,
                                              ParsingIndex=None, GenerateTree=True,
                                              use_pred_segmentation=True)
    return edu[0], (spans[0][0] if spans and spans[0] else "")

def relations_from_span(span):                               # one relation per parsing group
    prof = Counter()
    for grp in re.findall(r"\([^()]*\)", span):
        for r in {m for m in re.findall(r"=([\w-]+):", grp) if m != "span"}:
            prof[DM2NIS.get(r, "Other")] += 1
    return prof

def nisaba_narrative(t):                                     # interleaved narrative only
    i = t.find("## Interleaved Description")
    return t[i:] if i >= 0 else t

# --- 5. corroborate on the eval tree if present, else the CURRENT model's two in-memory descriptions
#        (same scope as the lexical RST cell above). ---
ROOT = EVAL_ROOT if 'EVAL_ROOT' in globals() and os.path.isdir(EVAL_ROOT) else "."
pairs = []
for nis in sorted(glob.glob(os.path.join(ROOT, "**", "*_nisaba_description.txt"), recursive=True)):
    zs = nis.replace("_nisaba_description", "_zero-shot_description")
    if os.path.exists(zs):
        m = re.search(r"(model\d+)_(.+?)_nisaba_description", os.path.basename(nis))
        pairs.append(((m.group(1) if m else os.path.basename(nis)), (m.group(2) if m else "-"),
                      nisaba_narrative(open(nis, encoding="utf-8").read()),
                      open(zs, encoding="utf-8").read()))
if not pairs:                                                # linear single-model run
    pairs = [("current", "-", nisaba_narrative(nisaba_description), zeroshot_description)]

raw, rows, poolN, poolZ = {}, [], Counter(), Counter()
for model, fam, ntext, ztext in pairs:
    en, sn = parse_text(ntext); ez, sz = parse_text(ztext)
    pn, pz = relations_from_span(sn), relations_from_span(sz)
    poolN.update(pn); poolZ.update(pz)
    raw[f"{model}/{fam}"] = {"nisaba":   {"edus": len(en), "span": sn, "profile": dict(pn)},
                             "zeroshot": {"edus": len(ez), "span": sz, "profile": dict(pz)}}
    ntN = sum(v for k, v in pn.items() if k in NONTEMPORAL)
    ntZ = sum(v for k, v in pz.items() if k in NONTEMPORAL)
    rows.append([model, fam, len(en), len(ez), sum(pn.values()), sum(pz.values()), ntN, ntZ])
    print(f"{model}/{fam}: EDU {len(en)}/{len(ez)}  nontemp {ntN}/{ntZ}")

# --- 6. write outputs ---
json.dump(raw, open("dmrst_raw.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
with open("dmrst_results.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["model", "family", "edu_nisaba", "edu_zeroshot", "rel_nisaba",
                "rel_zeroshot", "nontemporal_nisaba", "nontemporal_zeroshot"])
    w.writerows(rows)

# --- 7. headline corroboration ---
if rows:
    mN = sum(r[6] for r in rows) / len(rows); mZ = sum(r[7] for r in rows) / len(rows)
    print(f"\nMEAN non-temporal rels (Contrast+Condition+Cause): Nisaba {mN:.1f} vs zero-shot {mZ:.1f}")
    print("Pooled DMRST relation distribution (mapped to Nisaba relations):")
    for rel in sorted(set(poolN) | set(poolZ), key=lambda k: -(poolN[k] + poolZ[k])):
        print(f"  {rel:<22} Nisaba {poolN[rel]:>4}  zero-shot {poolZ[rel]:>4}")


### Comprehension QA — functional understandability metric

Understandability is assessed by (i) **FRE** (readability, below) and (ii) this **Comprehension QA**: a reader answers yes/no questions from the description ALONE, and the answers are graded against a **deterministic ground truth derived from the MP-Declare model** (constraint semantics verified against declare4py's conformance checker — in particular the Precedence direction). It is non-circular and generator-independent — it measures whether a reader can recover the process facts from the description. Scored with **balanced accuracy** (neutralises the yes/no base rate) over a **2-reader cross-vendor panel**. A human study remains the ultimate arbiter of understandability and is left as future work.

In [ ]:
# ==========================================================================
# Comprehension QA -- functional understandability (deterministic ground truth from the MP-Declare
# model; semantics verified vs declare4py conformance). Reader answers from the DESCRIPTION ONLY;
# scored with balanced accuracy over a 2-reader cross-vendor panel. Deterministic ground truth, so non-circular.
# ==========================================================================

# Reader panel: 1 closed (OpenAI) + 1 open (Z-AI), reasoning off, each stronger than every generator;
# QA balanced-accuracy is averaged over both. Defined once in the model-configuration cell (READERS).
READER_MODELS = list(READERS.values())

def _qa_constraints(decl):
    out=[]
    for line in decl.splitlines():
        m=re.match(r"^\s*([A-Za-z][A-Za-z \-]*?)\s*\[(.*?)\]", line)
        if not m: continue
        head=m.group(1).strip()
        if head.lower() in ("activity","bind"): continue
        out.append((head,[a.strip() for a in m.group(2).split(",") if a.strip()]))
    return out

def build_qa(decl):
    """Deterministic yes/no comprehension questions; ground truth from constraint semantics.
    Only RELATIONS/PROHIBITIONS (discriminative); reverse-direction distractors balance yes/no.
    Precedence direction VERIFIED against declare4py conformance: Precedence[X,Y] => X precedes Y."""
    qa=[]; add=lambda q,a,t: qa.append({"q":q,"a":a,"type":t})
    cons=_qa_constraints(decl)
    fwd={(a[0],a[1]) for ct,a in cons if len(a)>1 and ct.lower() in
         ("response","alternate response","chain response","succession","alternate succession","chain succession")}
    prec={(a[0],a[1]) for ct,a in cons if len(a)>1 and ct.lower() in
          ("precedence","alternate precedence","chain precedence")}
    for ctype,acts in cons:
        t=ctype.lower(); A=acts[0] if acts else None; B=acts[1] if len(acts)>1 else None
        if t=="absence" and A:
            add(f"Is '{A}' allowed to occur in the process?","no",ctype)
        elif t in ("response","alternate response","chain response") and A and B:
            add(f"If '{A}' occurs, must '{B}' occur afterwards?","yes",ctype)
            if A!=B and (B,A) not in fwd: add(f"If '{B}' occurs, must '{A}' occur afterwards?","no",ctype+"-rev")
        elif t in ("succession","alternate succession","chain succession") and A and B:
            add(f"If '{A}' occurs, must '{B}' occur after it?","yes",ctype)
            if A!=B and (B,A) not in fwd: add(f"If '{B}' occurs, must '{A}' occur after it?","no",ctype+"-rev")
        elif t in ("precedence","alternate precedence") and A and B:
            add(f"Can '{B}' occur without '{A}' having occurred before it?","no",ctype)
            if A!=B and (B,A) not in prec: add(f"Can '{A}' occur without '{B}' having occurred before it?","yes",ctype+"-rev")
        elif t=="chain precedence" and A and B:
            add(f"Can '{B}' occur without '{A}' occurring immediately before it?","no",ctype)
        elif t in ("not precedence","not chain precedence") and A and B:
            add(f"Is '{B}' allowed to occur after '{A}' has already occurred earlier?","no",ctype)
        elif t=="responded existence" and A and B:
            add(f"If '{A}' occurs, must '{B}' also occur in the same case?","yes",ctype)
        elif t=="co-existence" and A and B:
            add(f"If '{A}' occurs, must '{B}' also occur in the same case?","yes",ctype)
        elif t=="choice" and A and B:
            add(f"Must at least one of '{A}' or '{B}' occur?","yes",ctype)
        elif t=="exclusive choice" and A and B:
            add(f"Can both '{A}' and '{B}' occur in the same case?","no",ctype)
        elif t=="not co-existence" and A and B:
            add(f"Can both '{A}' and '{B}' occur in the same case?","no",ctype)
        elif t in ("not responded existence","not response","not succession") and A and B:
            add(f"If '{A}' occurs, is '{B}' allowed to occur in response/relation to it?","no",ctype)
        elif t in ("not chain succession","not chain response") and A and B:
            if A==B: add(f"Can two '{A}' activities occur immediately one after another (back-to-back)?","no",ctype)
            else:    add(f"Can '{B}' occur immediately after '{A}'?","no",ctype)
    seen=set(); uniq=[]
    for x in qa:
        if x["q"] not in seen: seen.add(x["q"]); uniq.append(x)
    return uniq

def balanced_accuracy(qa, answers):
    tot={"yes":0,"no":0}; cor={"yes":0,"no":0}
    for x,a in zip(qa,answers):
        if x["a"] in tot:
            tot[x["a"]]+=1
            if a==x["a"]: cor[x["a"]]+=1
    cls=[c for c in ("yes","no") if tot[c]>0]
    return None if len(cls)<2 else round(sum(cor[c]/tot[c] for c in cls)/len(cls),3)

class _QAAnswers(BaseModel):
    answers: List[Literal["yes","no","unknown"]]

def comprehension_qa(description, qa, reader_models=READER_MODELS, return_detail=False):
    """Reader answers from the DESCRIPTION ONLY; balanced accuracy averaged over the reader panel.
    With return_detail=True, also returns the per-reader balanced accuracies (dict)."""
    qtext="\n".join(f"{i+1}. {x['q']}" for i,x in enumerate(qa))
    msg=[{"role":"user","content":
          "You are given ONLY a natural-language description of a business process (no formal model). "
          f"Answer ALL {len(qa)} yes/no questions IN ORDER using only what the description states or clearly "
          "implies; answer 'unknown' if the description does not say.\n\nDESCRIPTION:\n"+description+
          "\n\nQUESTIONS:\n"+qtext}]
    accs={}; raw={}
    for rm in reader_models:
        ans=structured_output(messages=msg, response_format=_QAAnswers, model=rm).choices[0].message.parsed.answers
        raw[rm]=ans
        ba=balanced_accuracy(qa, ans)
        if ba is not None: accs[rm]=ba
    panel=round(sum(accs.values())/len(accs),3) if accs else None
    return (panel, {"ba": accs, "answers": raw}) if return_detail else panel

# --- run on the current model's descriptions (nisaba vs zero-shot) ---
_decl = "\n".join(declare_model.declare_model_lines)
_qa = build_qa(_decl)
_nis_narr = nisaba_description[nisaba_description.find("## Interleaved Description"):] \
            if "## Interleaved Description" in nisaba_description else nisaba_description
print(f"Comprehension QA: {len(_qa)} questions "
      f"({sum(1 for x in _qa if x['a']=='yes')} yes / {sum(1 for x in _qa if x['a']=='no')} no)")
print("Nisaba    balanced QA:", comprehension_qa(nisaba_description, _qa))
print("Zero-shot balanced QA:", comprehension_qa(zeroshot_description, _qa))


## Batch — industrial-scale generation & evaluation

The walkthrough above runs the pipeline on **one** model (the running example) with the default
generator, so every step's output is visible. This section runs the **same** pipeline over the whole
evaluation set (`BATCH_MODELS`) and **every** generator (`BATCH_GENERATORS`) — configured in the
*Evaluation set* cell above — then applies every instrument and saves the results.

Outputs mirror each model's source folder under `BATCH_OUT`, with filenames prefixed by the generator, so
one model reconstructed by four generators sits side by side. Generation is **resumable** (a combo whose
reconstructions already exist is skipped), so the batch can be stopped and continued.

In [ ]:
# === Batch: the single-model recipe as a function ===
# run_pipeline is exactly Steps 1-3 + baseline + reconstruction from above, wrapped so the evaluation set
# can be processed at scale with the IDENTICAL pipeline. The generator is selected with set_active_model
# (picked up by the LLM primitives) and passed to the CrewAI narrative; reconstruction uses the same
# generator (a description is decoded by the model that wrote it, as in the single-model demo).
import os, glob, json, shutil

def _combo_meta(decl_path):
    """Map an evaluation-set .decl path (.../<Source>/<Batch>/model<ID>/model<ID>.decl) to its identity."""
    p = decl_path.replace("\\", "/").split("/")
    return {"provider": p[-4], "batch": p[-3], "id": os.path.splitext(os.path.basename(decl_path))[0],
            "src_dir": os.path.dirname(decl_path)}

def run_pipeline(decl_string, gen_model):
    set_active_model(gen_model)                         # generation + reconstruction route to this generator
    dm = ExtendedDeclareModel().parse_from_string(decl_string)
    canon = "\n".join(dm.declare_model_lines)
    # Step 2 - intermediary (tool calls; structured-output fallback for providers without native tools)
    msgs = [{"role": "system", "content": decltotextassistant},
            {"role": "user", "content": "New model to generate description of activities, attributes, binds "
                                         "and constraints based in the template and examples: \n" + canon}]
    with log_context(step="generation.intermediary"):
        resp = chat_completion_request(messages=msgs, tools=toolsdescription, tool_choice="auto", temperature=0)
    tcs = resp.choices[0].message.tool_calls
    if not tcs:
        class _CItem(BaseModel):
            name: str
            description: str
        class _Intermediary(BaseModel):
            activities:  List[_CItem] = Field(default_factory=list)
            attributes:  List[_CItem] = Field(default_factory=list)
            binds:       List[_CItem] = Field(default_factory=list)
            constraints: List[_CItem] = Field(default_factory=list)
        with log_context(step="generation.intermediary.fallback"):
            parsed = structured_output(messages=msgs, response_format=_Intermediary,
                                       temperature=0).choices[0].message.parsed
        tcs = [SimpleNamespace(function=SimpleNamespace(arguments=parsed.model_dump_json()))]
    intermediary, _ = process_intermediate_descriptions(tcs)
    # Step 3 - agentic narrative (same crew_narrative as the demo)
    with log_context(step="generation.narrative"):
        narrative = crew_narrative(intermediary, gen_model)
    nisaba_desc = f"#MP-Declare Model\n{intermediary}\n## Interleaved Description\n{narrative}"
    # Baseline - zero-shot
    with log_context(step="zeroshot"):
        zeroshot = chat_completion_request(
            messages=[{"role": "user", "content": "Generate a natural language description of the following "
                       "Multi-Perspective Declare model:\n" + canon}], temperature=0).choices[0].message.content
    # Reconstruction from BOTH final descriptions
    with log_context(step="reconstruction"):
        recon = {tag: reconstruct_from_description(d)
                 for tag, d in [("nisaba", nisaba_desc), ("zeroshot", zeroshot)]}
    return {"canon": canon, "nisaba_description": nisaba_desc,
            "zeroshot_description": zeroshot, "reconstructed": recon}

In [ ]:
# === Batch generation: run_pipeline over every (model x generator), resumable ===
os.makedirs(BATCH_OUT, exist_ok=True)
_n_done = _n_run = 0
for _decl_path in BATCH_MODELS:
    _meta = _combo_meta(_decl_path)
    _dst = os.path.join(BATCH_OUT, _meta["provider"], _meta["batch"], _meta["id"])
    os.makedirs(_dst, exist_ok=True)
    _decl = open(_decl_path, encoding="utf-8").read()
    for _ext in ("decl", "json"):                       # copy the source model in (baseline for the metrics)
        _s = os.path.join(_meta["src_dir"], _meta["id"] + "." + _ext)
        _d = os.path.join(_dst, _meta["id"] + "." + _ext)
        if os.path.exists(_s) and not os.path.exists(_d):
            shutil.copy2(_s, _d)
    for _gen_name, _gen_model in BATCH_GENERATORS.items():
        _pre = os.path.join(_dst, f"{_meta['id']}_{_gen_name}")
        if os.path.exists(_pre + "_reconstructed.decl") and os.path.exists(_pre + "_zeroshot_reconstructed.decl"):
            _n_done += 1
            continue                                    # resumable: this combo is already complete
        print(f"[{_meta['id']}/{_gen_name}] generating ...", flush=True)
        with log_context(model=_meta["id"], provider=_gen_name, source=_meta["provider"], batch=_meta["batch"]):
            _r = run_pipeline(_decl, _gen_model)
        open(_pre + "_nisaba_description.txt", "w", encoding="utf-8").write(_r["nisaba_description"])
        open(_pre + "_zero-shot_description.txt", "w", encoding="utf-8").write(_r["zeroshot_description"])
        for _tag, _suf in [("nisaba", "_reconstructed"), ("zeroshot", "_zeroshot_reconstructed")]:
            _m = _r["reconstructed"][_tag]
            json.dump(_m.model_dump_json(), open(_pre + _suf + ".json", "w", encoding="utf-8"), indent=4)
            open(_pre + _suf + ".decl", "w", encoding="utf-8").write(_m.convert_to_string())
        _n_run += 1
print(f"\nbatch generation: {_n_run} combos generated, {_n_done} already present -> {BATCH_OUT}")

In [ ]:
# === Batch evaluation: apply every instrument to the batch outputs and PERSIST every analysis input ===
import csv as _csv, math as _math
_tau = EMB_TAU if "EMB_TAU" in globals() else DEFAULT_TAU

# --- statistical helpers (also used by the derived-summary cell below) ---
def gwet_ac1(a, b, categories=None):
    """Gwet's AC1 inter-rater agreement (2 raters), robust to the answer-prevalence imbalance that makes
    Cohen's kappa paradoxical here."""
    n = min(len(a), len(b))
    if n == 0: return None
    a, b = a[:n], b[:n]
    cats = categories if categories is not None else sorted(set(a) | set(b))
    q = len(cats)
    if q < 2: return 1.0
    pa = sum(1 for x, y in zip(a, b) if x == y) / n
    pi = {k: (sum(1 for x in a if x == k) + sum(1 for x in b if x == k)) / (2 * n) for k in cats}
    pe = sum(pi[k] * (1 - pi[k]) for k in cats) / (q - 1)
    return 1.0 if pe >= 1 else round((pa - pe) / (1 - pe), 3)

def raw_agreement(a, b):
    n = min(len(a), len(b))
    return None if n == 0 else round(sum(1 for x, y in zip(a[:n], b[:n]) if x == y) / n, 3)

def cohens_d_paired(x, y):
    """Standardized paired effect size d_z = mean(x - y) / sd(x - y)."""
    d = [xi - yi for xi, yi in zip(x, y)]; n = len(d)
    if n < 2: return None
    m = sum(d) / n; sd = _math.sqrt(sum((di - m) ** 2 for di in d) / (n - 1))
    return None if sd == 0 else round(m / sd, 2)

def direction_consistency(x, y, cmp="lt"):
    """(count, n) of pairs where x < y (cmp='lt') or x > y ('gt')."""
    n = min(len(x), len(y))
    c = sum(1 for xi, yi in zip(x[:n], y[:n]) if (xi < yi if cmp == "lt" else xi > yi))
    return (c, n)

# Folder-walking instruments (same functions as the demo, pointed at the batch tree):
_cx = calculate_metrics(BATCH_OUT)
save_metrics_to_csv(_cx, BATCH_OUT, "complexity_metrics.csv")
_fre = process_folder(BATCH_OUT, os.path.join(BATCH_OUT, "readability_scores.csv"))
_fre_by_file = dict(zip(_fre["filename"], _fre["flesch_reading_ease"])) if len(_fre) else {}

# Per-combo instruments. Besides the wide summary (batch_results.csv) we persist the GRANULAR input each
# paper table needs: per-perspective Semantic Distance, per-family RST realisation, per-reader QA, and the
# inter-reader agreement that the QA summary reports.
_rows, _sd_rows, _rst_rows, _qa_rows, _qa_agree = [], [], [], [], []
_rst_pool = {}   # constraint type -> [nis_aligned, nis_mentioned, zs_aligned, zs_mentioned]
for _decl_path in BATCH_MODELS:
    _meta = _combo_meta(_decl_path)
    _dst = os.path.join(BATCH_OUT, _meta["provider"], _meta["batch"], _meta["id"])
    _orig_path = os.path.join(_dst, _meta["id"] + ".decl")
    if not os.path.exists(_orig_path):
        continue
    _orig = open(_orig_path, encoding="utf-8").read()
    _cons = _rst_constraints(_orig)
    _qa = build_qa(_orig)
    for _gen_name in BATCH_GENERATORS:
        _pre = os.path.join(_dst, f"{_meta['id']}_{_gen_name}")
        _np, _zp = _pre + "_nisaba_description.txt", _pre + "_zero-shot_description.txt"
        if not (os.path.exists(_np) and os.path.exists(_zp)):
            continue
        _nis = open(_np, encoding="utf-8").read()
        _zs = open(_zp, encoding="utf-8").read()
        _row = {"model": _meta["id"], "source": _meta["provider"], "generator": _gen_name}
        # Semantic Distance: Total in the summary + the 5 perspectives in semantic_distance.csv
        for _tag, _suf in [("nisaba", "_reconstructed.decl"), ("zeroshot", "_zeroshot_reconstructed.decl")]:
            _rp = _pre + _suf
            if os.path.exists(_rp):
                _sd = compute_semantic_distance(_orig, open(_rp, encoding="utf-8").read(), embedding_model, tau=_tau)
                _row["SD_" + _tag] = round(_sd["total"], 4)
                _sdr = {"model": _meta["id"], "generator": _gen_name, "recon_source": _tag,
                        "tau": _tau, "total": round(_sd["total"], 4)}
                for _k, _v in _sd["perspectives"].items():
                    _sdr[_k] = None if _v is None else round(_v, 4)
                _sd_rows.append(_sdr)
        # RST: aligned_rate/coverage in the summary; pooled per-family realisation for tab:rst_relation_metric
        _nis_narr = _nis[_nis.find("## Interleaved Description"):] if "## Interleaved Description" in _nis else _nis
        _rn = rst_relation_metric(_nis_narr, _cons)
        _rz = rst_relation_metric(_zs, _cons)
        _row["RST_nisaba"] = round(_rn["aligned_rate"], 3)
        _row["RST_zeroshot"] = round(_rz["aligned_rate"], 3)
        _rst_rows.append({"model": _meta["id"], "generator": _gen_name, "n_constraints": _rn["n"],
                          "coverage_nisaba": round(_rn["coverage"], 3), "coverage_zeroshot": round(_rz["coverage"], 3),
                          "aligned_nisaba": round(_rn["aligned_rate"], 3), "aligned_zeroshot": round(_rz["aligned_rate"], 3)})
        for _ct in set(_rn["per_type"]) | set(_rz["per_type"]):
            _slot = _rst_pool.setdefault(_ct, [0, 0, 0, 0])     # per_type value = (mentioned, aligned)
            _m, _a = _rn["per_type"].get(_ct, (0, 0)); _slot[0] += _a; _slot[1] += _m
            _m, _a = _rz["per_type"].get(_ct, (0, 0)); _slot[2] += _a; _slot[3] += _m
        # Readability from the folder walk
        _row["FRE_nisaba"] = _fre_by_file.get(os.path.basename(_np))
        _row["FRE_zeroshot"] = _fre_by_file.get(os.path.basename(_zp))
        # Comprehension QA: panel mean in the summary, per-reader in qa_detail.csv, inter-reader agreement
        # (Gwet AC1 + raw) in qa_agreement.csv -- computed on the raw per-question answers of the two readers.
        _qn, _qn_det = comprehension_qa(_nis, _qa, return_detail=True)
        _qz, _qz_det = comprehension_qa(_zs, _qa, return_detail=True)
        _row["QA_nisaba"] = _qn
        _row["QA_zeroshot"] = _qz
        for _rdr in set(_qn_det["ba"]) | set(_qz_det["ba"]):
            _qa_rows.append({"model": _meta["id"], "generator": _gen_name, "reader": _rdr,
                             "qa_nisaba": _qn_det["ba"].get(_rdr), "qa_zeroshot": _qz_det["ba"].get(_rdr)})
        _an = list(_qn_det["answers"].values()); _az = list(_qz_det["answers"].values())
        if len(_an) >= 2 and len(_az) >= 2:
            _qa_agree.append({"model": _meta["id"], "generator": _gen_name,
                              "ac1_nisaba": gwet_ac1(_an[0], _an[1], ["yes", "no", "unknown"]),
                              "raw_nisaba": raw_agreement(_an[0], _an[1]),
                              "ac1_zeroshot": gwet_ac1(_az[0], _az[1], ["yes", "no", "unknown"]),
                              "raw_zeroshot": raw_agreement(_az[0], _az[1])})
        _rows.append(_row)
        print(f"  {_meta['id']}/{_gen_name}: SD n/z={_row.get('SD_nisaba')}/{_row.get('SD_zeroshot')}  "
              f"RST n/z={_row['RST_nisaba']}/{_row['RST_zeroshot']}  QA n/z={_row['QA_nisaba']}/{_row['QA_zeroshot']}")

# Persist the summary AND every table's granular input:
batch_results = pd.DataFrame(_rows)
batch_results.to_csv(os.path.join(BATCH_OUT, "batch_results.csv"), index=False)
pd.DataFrame(_sd_rows).to_csv(os.path.join(BATCH_OUT, "semantic_distance.csv"), index=False)
pd.DataFrame(_rst_rows).to_csv(os.path.join(BATCH_OUT, "rst_metrics.csv"), index=False)
pd.DataFrame(_qa_rows).to_csv(os.path.join(BATCH_OUT, "qa_detail.csv"), index=False)
pd.DataFrame(_qa_agree).to_csv(os.path.join(BATCH_OUT, "qa_agreement.csv"), index=False)
with open(os.path.join(BATCH_OUT, "rst_relation_families.csv"), "w", newline="", encoding="utf-8") as _f:
    _w = _csv.writer(_f)
    _w.writerow(["constraint_type", "prescribed_relation",
                 "nisaba_aligned", "nisaba_mentioned", "zeroshot_aligned", "zeroshot_mentioned"])
    for _ct in sorted(_rst_pool, key=lambda k: -_rst_pool[k][1]):
        _p = _rst_pool[_ct]
        _w.writerow([_ct, RST_CONSTRAINT_RELATION.get(_ct, "?"), _p[0], _p[1], _p[2], _p[3]])
print(f"\n{len(batch_results)} combos evaluated -> batch_results.csv "
      f"(+ semantic_distance.csv, rst_metrics.csv, rst_relation_families.csv, qa_detail.csv, qa_agreement.csv)")
if len(batch_results):
    print(batch_results.to_string(index=False))

In [ ]:
# === Batch: full DMRST distribution (optional) + cost report ===
import csv as _csv

# DMRST neural corroboration over the batch (reuses the parser loaded in the DMRST cell). Persists the FULL
# per-relation distribution (not only the non-temporal sum); dmrst_pooled.csv is the source of the DMRST table.
if "parse_text" in globals():
    _dpoolN, _dpoolZ, _drows = Counter(), Counter(), []
    for _nisf in sorted(glob.glob(os.path.join(BATCH_OUT, "**", "*_nisaba_description.txt"), recursive=True)):
        _zsf = _nisf.replace("_nisaba_description", "_zero-shot_description")
        if not os.path.exists(_zsf):
            continue
        _en, _sn = parse_text(nisaba_narrative(open(_nisf, encoding="utf-8").read()))
        _ez, _sz = parse_text(open(_zsf, encoding="utf-8").read())
        _pn, _pz = relations_from_span(_sn), relations_from_span(_sz)
        _dpoolN.update(_pn); _dpoolZ.update(_pz)
        _drows.append({"file": os.path.basename(_nisf), "edu_nisaba": len(_en), "edu_zeroshot": len(_ez),
                       "nontemporal_nisaba":   sum(v for k, v in _pn.items() if k in NONTEMPORAL),
                       "nontemporal_zeroshot": sum(v for k, v in _pz.items() if k in NONTEMPORAL)})
    if _drows:
        pd.DataFrame(_drows).to_csv(os.path.join(BATCH_OUT, "dmrst_batch.csv"), index=False)
        with open(os.path.join(BATCH_OUT, "dmrst_pooled.csv"), "w", newline="", encoding="utf-8") as _f:
            _w = _csv.writer(_f)
            _w.writerow(["relation", "nisaba", "zeroshot"])
            for _rel in sorted(set(_dpoolN) | set(_dpoolZ), key=lambda k: -(_dpoolN[k] + _dpoolZ[k])):
                _w.writerow([_rel, _dpoolN[_rel], _dpoolZ[_rel]])
        _mN = sum(d["nontemporal_nisaba"] for d in _drows) / len(_drows)
        _mZ = sum(d["nontemporal_zeroshot"] for d in _drows) / len(_drows)
        print(f"DMRST: mean non-temporal  Nisaba {_mN:.1f} vs zero-shot {_mZ:.1f}  "
              f"({len(_drows)} combos) -> dmrst_pooled.csv")

# Cost report: the provenance log (now carrying token usage) x OpenRouter pricing -> run_meta.json.
PRICING = {  # per 1M tokens (in, out), OpenRouter -- 4 generators + 2 readers
    "openrouter/anthropic/claude-sonnet-5": (2.0, 10.0), "openrouter/x-ai/grok-4.20": (1.25, 2.5),
    "openrouter/deepseek/deepseek-v4-flash": (0.084, 0.168), "openrouter/z-ai/glm-5.2": (0.392, 1.232),
    "openrouter/openai/gpt-5.6-terra": (2.5, 15.0), "openrouter/moonshotai/kimi-k2.5": (0.375, 2.025),
}
_usage, _total = {}, 0.0
if os.path.exists(LLM_CALL_LOG):
    for _line in open(LLM_CALL_LOG, encoding="utf-8"):
        try:
            _r = json.loads(_line); _u = _r.get("usage") or {}
            _a = _usage.setdefault(_r.get("model"), {"prompt": 0, "completion": 0, "calls": 0})
            _a["prompt"] += _u.get("prompt_tokens") or 0
            _a["completion"] += _u.get("completion_tokens") or 0
            _a["calls"] += 1
        except Exception:
            pass
for _mdl, _a in _usage.items():
    _pi, _po = PRICING.get(_mdl, (0, 0))
    _a["cost_usd"] = round(_a["prompt"] / 1e6 * _pi + _a["completion"] / 1e6 * _po, 4)
    _total += _a["cost_usd"]
_meta = {"generators": dict(BATCH_GENERATORS), "readers": READERS, "n_models": len(BATCH_MODELS),
         "usage_by_model": _usage, "est_cost_usd": round(_total, 3)}
json.dump(_meta, open(os.path.join(BATCH_OUT, "run_meta.json"), "w", encoding="utf-8"), indent=2)
print(f"cost: est ${_total:.2f} over {sum(a['calls'] for a in _usage.values())} logged calls -> run_meta.json")

In [ ]:
# === Derived summary statistics: the numbers behind the paper's summary tables -> derived_summary.json ===
import csv as _csv
_der = {}

# Complexity: exact-reconstruction rate + max |Delta| per metric (source vs Nisaba reconstruction)
_METRICS = [("Size", calculate_size_metric), ("Density", calculate_density_metric),
            ("Separability", calculate_separability_metric), ("Constraint variability", calculate_constraint_variability_metric)]
_cxagg = {m: {"exact": 0, "n": 0, "max": 0.0} for m, _ in _METRICS}
for _decl_path in BATCH_MODELS:
    _meta = _combo_meta(_decl_path)
    _dst = os.path.join(BATCH_OUT, _meta["provider"], _meta["batch"], _meta["id"])
    _sj = os.path.join(_dst, _meta["id"] + ".json")
    if not os.path.exists(_sj):
        continue
    try: _src = json.loads(json.loads(open(_sj, encoding="utf-8").read()))
    except Exception: continue
    for _gen in BATCH_GENERATORS:
        _rj = os.path.join(_dst, f"{_meta['id']}_{_gen}_reconstructed.json")
        if not os.path.exists(_rj):
            continue
        try: _rec = json.loads(json.loads(open(_rj, encoding="utf-8").read()))
        except Exception: continue
        for _mn, _fn in _METRICS:
            _d = abs(_fn(_src) - _fn(_rec))
            _cxagg[_mn]["n"] += 1
            _cxagg[_mn]["exact"] += (1 if _d == 0 else 0)
            _cxagg[_mn]["max"] = max(_cxagg[_mn]["max"], _d)
_der["complexity"] = {m: {"exact": f"{v['exact']}/{v['n']}", "max_abs_delta": round(v["max"], 4)}
                      for m, v in _cxagg.items()}

# Semantic Distance: fraction at 0 + max, per perspective (Nisaba reconstruction)
_sd_by = {}
_sdf = os.path.join(BATCH_OUT, "semantic_distance.csv")
if os.path.exists(_sdf):
    for _r in _csv.DictReader(open(_sdf, encoding="utf-8")):
        if _r.get("recon_source") != "nisaba":
            continue
        for _k in ["control_flow", "temporal", "data_conditions", "binds", "attr_domains", "total"]:
            _v = _r.get(_k)
            if _v in (None, "", "None"):
                continue
            _v = float(_v); _s = _sd_by.setdefault(_k, {"zero": 0, "n": 0, "max": 0.0})
            _s["n"] += 1; _s["zero"] += (1 if _v == 0 else 0); _s["max"] = max(_s["max"], _v)
_der["semantic_distance"] = {k: {"preserved": f"{v['zero']}/{v['n']}", "max": round(v["max"], 4)}
                             for k, v in _sd_by.items()}

# RST: per-family realised rate + gap (Nisaba - zero-shot)
_rst_gap = []
_rf = os.path.join(BATCH_OUT, "rst_relation_families.csv")
if os.path.exists(_rf):
    for _r in _csv.DictReader(open(_rf, encoding="utf-8")):
        _nm, _zm = int(_r["nisaba_mentioned"]), int(_r["zeroshot_mentioned"])
        _nr = int(_r["nisaba_aligned"]) / _nm if _nm else 0.0
        _zr = int(_r["zeroshot_aligned"]) / _zm if _zm else 0.0
        _rst_gap.append({"family": _r["constraint_type"], "prescribed": _r["prescribed_relation"],
                         "nisaba_rate": round(_nr, 3), "zeroshot_rate": round(_zr, 3), "gap": round(_nr - _zr, 3)})
_der["rst_gap"] = _rst_gap

# FRE: paired effect size d_z + directional consistency (Nisaba lower); QA: AC1/raw + Nisaba-superior
_fn_, _fz_, _qn_, _qz_ = [], [], [], []
_brf = os.path.join(BATCH_OUT, "batch_results.csv")
if os.path.exists(_brf):
    for _r in _csv.DictReader(open(_brf, encoding="utf-8")):
        try: _fn_.append(float(_r["FRE_nisaba"])); _fz_.append(float(_r["FRE_zeroshot"]))
        except Exception: pass
        try: _qn_.append(float(_r["QA_nisaba"])); _qz_.append(float(_r["QA_zeroshot"]))
        except Exception: pass
_der["fre"] = {"mean_diff": round(sum(z - n for z, n in zip(_fz_, _fn_)) / len(_fn_), 2) if _fn_ else None,
               "paired_d_z": cohens_d_paired(_fz_, _fn_),
               "nisaba_lower": "%d/%d" % direction_consistency(_fn_, _fz_, "lt")}

_ac = {c: [] for c in ["ac1_nisaba", "ac1_zeroshot", "raw_nisaba", "raw_zeroshot"]}
_qaf = os.path.join(BATCH_OUT, "qa_agreement.csv")
if os.path.exists(_qaf):
    for _r in _csv.DictReader(open(_qaf, encoding="utf-8")):
        for _c in _ac:
            _v = _r.get(_c)
            if _v not in (None, "", "None"):
                try: _ac[_c].append(float(_v))
                except Exception: pass
_mean = lambda Lst: round(sum(Lst) / len(Lst), 3) if Lst else None
_der["qa"] = {"balanced_acc_nisaba": _mean(_qn_), "balanced_acc_zeroshot": _mean(_qz_),
              "ac1_nisaba": _mean(_ac["ac1_nisaba"]), "ac1_zeroshot": _mean(_ac["ac1_zeroshot"]),
              "raw_nisaba": _mean(_ac["raw_nisaba"]), "raw_zeroshot": _mean(_ac["raw_zeroshot"]),
              "nisaba_superior": "%d/%d" % direction_consistency(_qz_, _qn_, "lt")}

json.dump(_der, open(os.path.join(BATCH_OUT, "derived_summary.json"), "w", encoding="utf-8"), indent=2)
print("=== Derived summary statistics (feed the paper's summary tables) -> derived_summary.json ===")
print(json.dumps(_der, indent=2))

### Dose–response: rhetorical realisation × reader outcomes

Does the **degree** to which a description realises the prescribed rhetorical relations predict how well readers understand it? This probe correlates (Spearman) the per-run lexical-RST `aligned_rate` with the reader panel's **balanced QA accuracy** and **reader-reconstruction distance** (recoverability). Two views: **pooled** across both approaches (320 points — uses the full treatment variance) and **within-Nisaba only** (160 points — the non-tautological test: the instruction is constant, only the degree of realisation varies). Reads `rst_metrics.csv` and the per-combo `*_understandability_v2.json` produced by the batch/reader stages (mirrored in `pipeline/dose_response.py`).

In [ ]:
# === Dose-response: per-run RST realisation vs reader comprehension / recoverability ===
import csv as _csv, json as _json, glob as _glob, os as _os
from scipy.stats import spearmanr as _spearmanr

_DR_ROOT = BATCH_OUT if _os.path.isdir(BATCH_OUT) else "Evaluation_SoSyM"   # batch tree (Colab) or repo layout
_rst = {(r["model"], r["family"]): r
        for r in _csv.DictReader(open(_os.path.join(_DR_ROOT, "rst_metrics.csv"), encoding="utf-8"))}
_und = {}
for _p in _glob.glob(_os.path.join(_DR_ROOT, "*", "*", "model*", "model*_*_understandability_v2.json")):
    _b = _os.path.basename(_p)
    _und[(_b.split("_")[0], _b.split("_")[1])] = _json.load(open(_p, encoding="utf-8"))

_pooled = {"al": [], "qa": [], "rec": []}; _within = {"al": [], "qa": [], "rec": []}
for _k, _r in _rst.items():
    _u = _und.get(_k)
    if not _u: continue
    for _t in ("nisaba", "zeroshot"):
        _al = _r.get(f"aligned_{_t}"); _qa = _u.get(f"qa_{_t}"); _rc = _u.get(f"recon_{_t}")
        if _al in (None, "") or _qa is None or _rc is None: continue
        _al = float(_al)
        _pooled["al"].append(_al); _pooled["qa"].append(_qa); _pooled["rec"].append(_rc)
        if _t == "nisaba":
            _within["al"].append(_al); _within["qa"].append(_qa); _within["rec"].append(_rc)

def _dr_report(_d, _label):
    _r1, _p1 = _spearmanr(_d["al"], _d["qa"]); _r2, _p2 = _spearmanr(_d["al"], _d["rec"])
    print(f"{_label} (n={len(_d['al'])}):")
    print(f"  aligned_rate vs balanced QA:    rho={_r1:+.3f}  p={_p1:.2g}")
    print(f"  aligned_rate vs recon distance: rho={_r2:+.3f}  p={_p2:.2g}   (negative = more rhetoric, more recoverable)")

_dr_report(_pooled, "POOLED (both approaches, 2 points/run)")
_dr_report(_within, "WITHIN-NISABA only (treatment constant; the non-tautological test)")
# marker density as an alternative dose variable (pooled)
_dp = [float(_rst[_k][f"density_{_t}"]) for _k in _rst if _k in _und for _t in ("nisaba", "zeroshot")
       if _rst[_k].get(f"density_{_t}") not in (None, "") and _und[_k].get(f"qa_{_t}") is not None]
_qp = [_und[_k][f"qa_{_t}"] for _k in _rst if _k in _und for _t in ("nisaba", "zeroshot")
       if _rst[_k].get(f"density_{_t}") not in (None, "") and _und[_k].get(f"qa_{_t}") is not None]
_r3, _p3 = _spearmanr(_dp, _qp)
print(f"\nmarker DENSITY vs balanced QA (pooled, n={len(_dp)}): rho={_r3:+.3f} p={_p3:.2g}")
# Caveats: correlational, modest effects; the pooled view conflates treatment with dose (within-Nisaba is the clean test).

## Artifacts (save & download)

In [ ]:
# File names
file_names = [
    "nisaba_description.txt",
    "zero-shot_description.txt"
]

# Zip file name
zip_file_name = "descriptions.zip"

# Create the zip file
with zipfile.ZipFile(zip_file_name, 'w') as zipf:
    for file_name in file_names:
        zipf.write(file_name)

print(f"{zip_file_name} created successfully.")

# Download the zip file
files.download(zip_file_name)

In [ ]:
# File names to delete
file_names = [
    "nisaba_description.txt",
    "zero-shot_description.txt",
    "descriptions.zip"  # Include the zip file as well
]

# Delete the files
for file_name in file_names:
    if os.path.exists(file_name):
        os.remove(file_name)
        print(f"{file_name} deleted successfully.")
    else:
        print(f"{file_name} does not exist. Skipping.")

print("All specified files have been deleted.")